<a href="https://colab.research.google.com/github/MWANIKID/Does-Architecture-Matter-for-Volatility-Forecasting/blob/main/Does_Architecture_Matter_for_Volatility_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================================
# COLAB SCRIPT
# Does Architecture Matter for Volatility Forecasting?
# Evidence Across Heterogeneous Equity Sectors
#
# PURPOSE
#   Main manuscript: compare out-of-sample volatility forecast accuracy of
#   LSTM, GRU, TCN, and Transformer models across NSE sectors.
#
# PRIMARY DATA CONSTRUCTION
#   - Daily firm panel, 2016-08-01 to 2026-07-31
#   - Lagged market-capitalization-weighted sector returns
#   - Historical returns are model inputs
#   - Future realized variance is the volatility forecast target
#
# PRIMARY HYPOTHESIS
#   H02: There is no statistically significant difference in the
#   out-of-sample volatility forecast accuracy among selected deep-learning
#   models across Nairobi Securities Exchange sectors.
#
# OUTPUT
#   All tables, figures, diagnostics, logs, and SI files are written to one
#   output directory and automatically zipped into ONE final ZIP file.
#
# IMPORTANT
#   1) Run first with QUICK_TEST=True to verify the pipeline.
#   2) Then set QUICK_TEST=False for the manuscript run.
#   3) Heavy SI modules are switchable because a full DL robustness battery
#      can exceed a normal free-Colab session.
# ======================================================================

# ----------------------------------------------------------------------
# 0. INSTALL PACKAGES
# ----------------------------------------------------------------------
import sys, subprocess, os, json, math, random, shutil, warnings, platform, time
from pathlib import Path

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["numpy", "pandas", "scipy", "scikit-learn", "statsmodels",
             "openpyxl", "matplotlib", "tensorflow>=2.15"])

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# 1. IMPORTS AND CONFIGURATION
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

try:
    from google.colab import files, drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None
    drive = None

# ---------------- USER SETTINGS ----------------
QUICK_TEST = False      # True = smoke test; False = manuscript-quality run
AUTO_DOWNLOAD_ZIP = True

# CRASH-SAFE / RESUME SETTINGS
USE_GOOGLE_DRIVE = True            # strongly recommended in Colab
RESUME_COMPLETED_JOBS = True        # automatically skip completed jobs
RESET_CURRENT_RUN = False           # set True ONLY when you intentionally want a fresh run
ENABLE_PROGRESS_ZIP = True          # periodic backup of current output tables/figures
SNAPSHOT_EVERY_N_COMPLETED_JOBS = 4
PROJECT_VERSION = "v2_crashsafe_20260911"

# Main manuscript design
PRIMARY_WEIGHTING = "value"     # lagged market-cap weighted
PRIMARY_MIN_STOCKS = 3          # primary multi-stock sector sample
PRIMARY_HORIZON = 5             # future 5-trading-day realized variance

# IMPORTANT: h=5 is NOT repeated here. It is already estimated in the primary run.
# The horizon-robustness block estimates only the additional horizons.
ROBUST_HORIZONS = [1, 22]
REPORT_HORIZONS = [1, 5, 22]

LOOKBACK = 60                   # past trading days of returns used as input

# Chronological sample split
TRAIN_END = "2022-07-31"
VALID_END = "2024-07-31"
# Test = dates after VALID_END through final available date.

# DL models
MODEL_NAMES = ["LSTM", "GRU", "TCN", "TRANSFORMER"]

# Repeated initializations
MAIN_SEEDS = [11, 29, 47, 71, 101]
ROBUST_SEEDS = [11, 47, 101]

# Training
MAX_EPOCHS = 120 if not QUICK_TEST else 8
PATIENCE = 12 if not QUICK_TEST else 2
BATCH_SIZE = 64
VERBOSE_FIT = 0
EPS = 1e-10

# Tuning: equal candidate budget for every architecture
N_TUNING_CONFIGS = 5 if not QUICK_TEST else 1
TUNING_SEED = 2026

# Statistical tests
DM_HAC_LAGS = None      # None -> horizon-dependent rule
MCS_ALPHA = 0.10
MCS_BOOTSTRAPS = 1000 if not QUICK_TEST else 100
MCS_BLOCK_LENGTH = 10

# Main-text / core robustness modules
RUN_CAPACITY_MATCH = True
RUN_HORIZON_ROBUSTNESS = True
RUN_REGIME_ROBUSTNESS = True
RUN_SIGNED_RETURN_ABLATION = True

# SI modules
RUN_EQUAL_WEIGHT_SI = True
RUN_STOCK_LEVEL_SI = False         # computationally heavier
RUN_PARKINSON_PROXY_SI = True
RUN_LOCAL_GLOBAL_SI = True
RUN_LEARNING_CURVES_SI = True
RUN_THIN_TRADING_SI = True

# Heavy SI modules: turn True for final SI production if Colab resources allow.
RUN_EXPANDING_WINDOW_SI = False
RUN_LOOKBACK_ABLATION_SI = False
RUN_EXTREME_RETURN_SI = True

# Robustness settings
LEARNING_CURVE_FRACTIONS = [0.40, 0.60, 0.80, 1.00]
LOOKBACK_GRID = [22, 60, 120]
THIN_TRADING_MAX_ZERO_RATE = 0.50
EXTREME_RETURN_WINSOR_Q = 0.005

# Sector series construction
MIN_ACTIVE_STOCKS_DAILY = 1
FFILL_WITHIN_LISTING_LIFE = True

# Main model-capacity target for architecture-fairness check
TARGET_PARAMETER_COUNT = 50000

# ----------------------------------------------------------------------
# PERSISTENT PROJECT STORAGE
# ----------------------------------------------------------------------
RUN_MODE = "quick_test" if QUICK_TEST else "full_run"

if IN_COLAB and USE_GOOGLE_DRIVE:
    print("Mounting Google Drive for crash-safe persistent storage...")
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/DL_Volatility_Forecasting_Project")
else:
    PROJECT_ROOT = Path("./DL_Volatility_Forecasting_Project")

DATA_ROOT = PROJECT_ROOT / "data"
RUN_ROOT = PROJECT_ROOT / "runs" / f"{PROJECT_VERSION}_{RUN_MODE}"
OUTPUT_ROOT = RUN_ROOT / "outputs"
CHECKPOINT_ROOT = RUN_ROOT / "checkpoints"
ZIP_ROOT = PROJECT_ROOT / "zips"

if RESET_CURRENT_RUN and RUN_ROOT.exists():
    print("RESET_CURRENT_RUN=True -> deleting:", RUN_ROOT)
    shutil.rmtree(RUN_ROOT)

for d in [DATA_ROOT, OUTPUT_ROOT, CHECKPOINT_ROOT, ZIP_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

for sub in [
    "00_manifest",
    "01_data_audit",
    "02_main_results",
    "03_robustness",
    "04_ablations",
    "05_SI",
    "06_figures",
    "07_logs",
    "08_predictions"
]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Persistent project root:", PROJECT_ROOT)
print("Current run root      :", RUN_ROOT)
print("Resume enabled        :", RESUME_COMPLETED_JOBS)
print("Additional horizons   :", ROBUST_HORIZONS, "(h=5 will be reused from primary)")

# ----------------------------------------------------------------------
# 2. REPRODUCIBILITY
# ----------------------------------------------------------------------
def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

set_all_seeds(TUNING_SEED)

# Reduce GPU-memory grabbing in Colab
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

# ----------------------------------------------------------------------
# 3. LOAD DATA
# ----------------------------------------------------------------------
EXPECTED_COLUMNS = [
    "Date", "Open", "High", "Low", "Close", "Volume", "Category", "Stock",
    "Bonus issue", "Issued Shares", "Market Capitalization (KES)"
]

def locate_or_upload_csv():
    """
    Crash-safe data locator.

    Priority:
      1) Persistent copy in Google Drive/project data folder.
      2) Exact filename already present in /content.
      3) Any single CSV in /content.
      4) Manual upload.

    Any newly found/uploaded CSV is copied to persistent DATA_ROOT so a
    Colab runtime reset does not require another upload.
    """
    preferred_name = "Final Master File_All variables 01082016-31072026.csv"
    persisted = DATA_ROOT / preferred_name

    if persisted.exists():
        print("Using persistent data copy:", persisted)
        return persisted

    local_exact = Path("/content") / preferred_name
    if local_exact.exists():
        shutil.copy2(local_exact, persisted)
        print("Copied input CSV to persistent storage:", persisted)
        return persisted

    if IN_COLAB:
        candidates = [
            p for p in Path("/content").glob("*.csv")
            if "diagnostic" not in p.name.lower()
        ]
        if len(candidates) == 1:
            shutil.copy2(candidates[0], persisted)
            print("Copied detected CSV to persistent storage:", persisted)
            return persisted

        print("Upload the master CSV file.")
        uploaded = files.upload()
        csvs = [Path("/content") / x for x in uploaded.keys()
                if x.lower().endswith(".csv")]
        if not csvs:
            raise FileNotFoundError("No CSV was uploaded.")
        # Use the uploaded file, but persist it under the stable expected name.
        shutil.copy2(csvs[0], persisted)
        print("Saved uploaded CSV persistently to:", persisted)
        return persisted

    # Local fallback
    candidates = list(Path(".").glob("*.csv"))
    if len(candidates) == 1:
        shutil.copy2(candidates[0], persisted)
        return persisted

    raise FileNotFoundError(
        "Could not locate the master CSV. Place it in the working directory "
        "or in DATA_ROOT."
    )

DATA_PATH = locate_or_upload_csv()
print("Using data:", DATA_PATH)

raw = pd.read_csv(DATA_PATH, low_memory=False)
missing_expected = [c for c in EXPECTED_COLUMNS if c not in raw.columns]
if missing_expected:
    raise ValueError(f"Missing expected columns: {missing_expected}")

# ----------------------------------------------------------------------
# 4. DATA CLEANING AND AUDIT
# ----------------------------------------------------------------------
df = raw.copy()
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date", "Stock", "Category"]).copy()

def to_num(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .str.replace(" ", "", regex=False)
         .replace({"nan": np.nan, "None": np.nan, "-": np.nan, "": np.nan}),
        errors="coerce"
    )

for c in ["Open", "High", "Low", "Close", "Volume",
          "Issued Shares", "Market Capitalization (KES)", "Bonus issue"]:
    df[c + "_num"] = to_num(df[c])

# Remove exact stock-date duplicates only after recording them
dup = df.duplicated(["Stock", "Date"], keep=False)
df.loc[dup].sort_values(["Stock", "Date"]).to_csv(
    OUTPUT_ROOT / "01_data_audit" / "duplicate_stock_dates.csv", index=False
)
df = df.sort_values(["Stock", "Date"]).drop_duplicates(["Stock", "Date"], keep="last")

# Basic OHLC validity
valid_ohlc = (
    (df["Open_num"] > 0) &
    (df["High_num"] > 0) &
    (df["Low_num"] > 0) &
    (df["Close_num"] > 0) &
    (df["High_num"] >= df[["Open_num", "Close_num", "Low_num"]].max(axis=1)) &
    (df["Low_num"] <= df[["Open_num", "Close_num", "High_num"]].min(axis=1))
)
df["valid_ohlc"] = valid_ohlc

# Corporate-action diagnostic flags based on share-count changes and large price moves.
df["prev_close_raw"] = df.groupby("Stock")["Close_num"].shift(1)
df["raw_log_return"] = np.log(df["Close_num"] / df["prev_close_raw"])
df["share_change"] = df.groupby("Stock")["Issued Shares_num"].pct_change(fill_method=None)
df["corporate_action_flag"] = (
    (df["share_change"].abs() >= 0.10) &
    (df["raw_log_return"].abs() >= 0.10)
) | df["Bonus issue_num"].notna()

corp_cols = ["Date", "Stock", "Category", "Close_num", "raw_log_return",
             "Issued Shares_num", "share_change", "Bonus issue", "corporate_action_flag"]
df.loc[df["corporate_action_flag"], corp_cols].to_csv(
    OUTPUT_ROOT / "01_data_audit" / "corporate_action_flags.csv", index=False
)

# Dataset summary
audit_summary = pd.DataFrame({
    "item": [
        "rows_raw", "rows_clean_stock_date", "unique_dates", "unique_stocks",
        "unique_sectors", "date_min", "date_max", "ohlc_invalid_rows",
        "corporate_action_flag_rows"
    ],
    "value": [
        len(raw), len(df), df["Date"].nunique(), df["Stock"].nunique(),
        df["Category"].nunique(), str(df["Date"].min().date()),
        str(df["Date"].max().date()), int((~df["valid_ohlc"]).sum()),
        int(df["corporate_action_flag"].sum())
    ]
})
audit_summary.to_csv(OUTPUT_ROOT / "01_data_audit" / "dataset_summary.csv", index=False)

sector_composition = (
    df.groupby("Category")["Stock"].nunique()
      .sort_values(ascending=False).rename("n_stocks").reset_index()
)
sector_composition.to_csv(
    OUTPUT_ROOT / "01_data_audit" / "sector_composition.csv", index=False
)

stock_coverage = (
    df.groupby(["Category", "Stock"])
      .agg(first_date=("Date", "min"),
           last_date=("Date", "max"),
           n_raw_obs=("Date", "size"))
      .reset_index()
)
stock_coverage.to_csv(
    OUTPUT_ROOT / "01_data_audit" / "stock_coverage.csv", index=False
)

# ----------------------------------------------------------------------
# 5. CREATE A DAILY STOCK PANEL ON THE EXCHANGE CALENDAR
# ----------------------------------------------------------------------
calendar = pd.Index(sorted(df["Date"].unique()), name="Date")

def build_stock_panel(source_df):
    """
    Reindex each stock to the union exchange calendar.
    Prices and market cap are forward-filled only between the stock's
    first and last raw observation. Missing trading days therefore
    generate stale-price zero returns, which are later stress-tested.
    """
    pieces = []

    for stock, g in source_df.groupby("Stock", sort=False):
        g = g.sort_values("Date").set_index("Date")
        first, last = g.index.min(), g.index.max()
        sector = g["Category"].dropna().iloc[-1]

        z = g.reindex(calendar)
        z["Stock"] = stock
        z["Category"] = sector
        z["raw_observed"] = z["Close_num"].notna().astype(int)

        active = (z.index >= first) & (z.index <= last)
        for c in ["Open_num", "High_num", "Low_num", "Close_num",
                  "Issued Shares_num", "Market Capitalization (KES)_num"]:
            if c in z:
                z.loc[active, c] = z.loc[active, c].ffill()

        # On an unobserved day within listing life, set stale OHLC equal to
        # carried close. This allows explicit thin-trading diagnostics.
        miss_trade = active & (z["raw_observed"] == 0)
        z.loc[miss_trade, "Open_num"] = z.loc[miss_trade, "Close_num"]
        z.loc[miss_trade, "High_num"] = z.loc[miss_trade, "Close_num"]
        z.loc[miss_trade, "Low_num"] = z.loc[miss_trade, "Close_num"]

        # Outside listing life -> NA
        keep_cols = ["Open_num", "High_num", "Low_num", "Close_num",
                     "Issued Shares_num", "Market Capitalization (KES)_num"]
        z.loc[~active, keep_cols] = np.nan

        z["active"] = active.astype(int)
        z["log_return"] = np.log(z["Close_num"] / z["Close_num"].shift(1))
        z.loc[z["active"] == 0, "log_return"] = np.nan

        # Parkinson variance proxy at stock level.
        hl = np.log(z["High_num"] / z["Low_num"])
        z["parkinson_var"] = (hl ** 2) / (4.0 * np.log(2.0))
        z.loc[(z["High_num"] <= 0) | (z["Low_num"] <= 0), "parkinson_var"] = np.nan

        # Zero-return / stale-price diagnostics
        z["zero_return"] = np.where(z["log_return"].notna(),
                                    (z["log_return"].abs() < 1e-14).astype(float),
                                    np.nan)
        z["raw_missing_within_life"] = np.where(active, 1 - z["raw_observed"], np.nan)
        pieces.append(z.reset_index())

    return pd.concat(pieces, ignore_index=True)

panel = build_stock_panel(df)

# Stock thin-trading audit
stock_thin = (
    panel.groupby(["Category", "Stock"])
         .agg(n_panel_obs=("log_return", "count"),
              zero_return_rate=("zero_return", "mean"),
              missing_trade_rate=("raw_missing_within_life", "mean"),
              first_date=("Date", "min"),
              last_date=("Date", "max"))
         .reset_index()
)
stock_thin.to_csv(
    OUTPUT_ROOT / "01_data_audit" / "stock_thin_trading_audit.csv", index=False
)

sector_thin = (
    stock_thin.groupby("Category")
              .agg(n_stocks=("Stock", "nunique"),
                   median_zero_return_rate=("zero_return_rate", "median"),
                   mean_zero_return_rate=("zero_return_rate", "mean"),
                   median_missing_trade_rate=("missing_trade_rate", "median"))
              .reset_index()
)
sector_thin.to_csv(
    OUTPUT_ROOT / "01_data_audit" / "sector_thin_trading_audit.csv", index=False
)

PRIMARY_SECTORS = (
    sector_composition.loc[sector_composition["n_stocks"] >= PRIMARY_MIN_STOCKS, "Category"]
    .tolist()
)
ALL_SECTORS = sector_composition["Category"].tolist()

print("Primary sectors (>= %d stocks):" % PRIMARY_MIN_STOCKS, PRIMARY_SECTORS)

# ----------------------------------------------------------------------
# 6. SECTOR RETURN CONSTRUCTION
# ----------------------------------------------------------------------
def build_sector_series(panel_df, weighting="value", sectors=None):
    """
    Build sector returns.
    value: lagged market-cap weights, so no contemporaneous weighting.
    equal: equal weights across active constituent stocks.
    Also constructs a value/equal-weighted average constituent Parkinson
    variance as an alternative volatility proxy for SI.
    """
    if sectors is None:
        sectors = sorted(panel_df["Category"].dropna().unique())

    p = panel_df[panel_df["Category"].isin(sectors)].copy()
    p = p.sort_values(["Stock", "Date"])
    p["lag_mcap"] = p.groupby("Stock")["Market Capitalization (KES)_num"].shift(1)

    out = []
    for sector, g in p.groupby("Category"):
        for dt, d in g.groupby("Date"):
            d = d[d["log_return"].notna()].copy()
            if d.empty:
                continue

            if weighting == "value":
                valid = d["lag_mcap"].gt(0) & d["lag_mcap"].notna()
                d = d.loc[valid]
                if d.empty:
                    continue
                w = d["lag_mcap"].to_numpy(dtype=float)
                w = w / w.sum()
            elif weighting == "equal":
                w = np.repeat(1.0 / len(d), len(d))
            else:
                raise ValueError("weighting must be 'value' or 'equal'")

            r = d["log_return"].to_numpy(dtype=float)
            sector_ret = np.sum(w * r)

            pv = d["parkinson_var"].to_numpy(dtype=float)
            good_pv = np.isfinite(pv)
            if good_pv.any():
                ww = w[good_pv]
                ww = ww / ww.sum()
                sector_pk = np.sum(ww * pv[good_pv])
            else:
                sector_pk = np.nan

            out.append({
                "Date": dt,
                "Category": sector,
                "sector_return": sector_ret,
                "sector_parkinson_var": sector_pk,
                "n_active": len(d),
                "largest_weight": float(w.max()) if len(w) else np.nan,
                "effective_n": float(1.0 / np.sum(w**2)) if len(w) else np.nan
            })

    ans = pd.DataFrame(out).sort_values(["Category", "Date"]).reset_index(drop=True)
    return ans

sector_value = build_sector_series(panel, "value", ALL_SECTORS)
sector_equal = build_sector_series(panel, "equal", ALL_SECTORS)

sector_value.to_csv(
    OUTPUT_ROOT / "01_data_audit" / "sector_series_value_weighted.csv", index=False
)
sector_equal.to_csv(
    OUTPUT_ROOT / "05_SI" / "sector_series_equal_weighted.csv", index=False
)

# Concentration diagnostics
concentration = (
    sector_value.groupby("Category")
                .agg(mean_largest_weight=("largest_weight", "mean"),
                     median_largest_weight=("largest_weight", "median"),
                     mean_effective_n=("effective_n", "mean"),
                     min_effective_n=("effective_n", "min"),
                     mean_active=("n_active", "mean"))
                .reset_index()
)
concentration.to_csv(
    OUTPUT_ROOT / "01_data_audit" / "sector_weight_concentration.csv", index=False
)

# ----------------------------------------------------------------------
# 7. FORECAST TARGETS AND FEATURE CONSTRUCTION
# ----------------------------------------------------------------------
def future_sum(x, h):
    # y_t = sum_{j=1}^h x_{t+j}
    s = pd.Series(x, dtype=float)
    return sum(s.shift(-j) for j in range(1, h + 1))

def add_targets(sector_df, h):
    z = sector_df.copy().sort_values("Date")
    z["sq_return"] = z["sector_return"] ** 2
    z[f"rv_h{h}"] = future_sum(z["sq_return"], h)
    z[f"pk_h{h}"] = future_sum(z["sector_parkinson_var"], h)
    return z

def feature_matrix(returns, mode="raw"):
    r = np.asarray(returns, dtype=float)
    if mode == "raw":
        return r[:, None]
    if mode == "posneg":
        return np.column_stack([np.maximum(r, 0), np.maximum(-r, 0)])
    if mode == "positive_only":
        return np.maximum(r, 0)[:, None]
    if mode == "negative_only":
        return np.maximum(-r, 0)[:, None]
    raise ValueError(mode)

def build_sequences_for_sector(sector_series, sector, h, lookback=LOOKBACK,
                               feature_mode="raw", target_proxy="rv",
                               train_end=TRAIN_END, valid_end=VALID_END):
    z = sector_series[sector_series["Category"] == sector].copy().sort_values("Date")
    z = add_targets(z, h)

    target_col = f"rv_h{h}" if target_proxy == "rv" else f"pk_h{h}"

    # Scale input using TRAINING period only.
    train_r = z.loc[z["Date"] <= pd.Timestamp(train_end), "sector_return"].dropna().values
    r_mean = np.mean(train_r)
    r_std = np.std(train_r, ddof=1)
    if not np.isfinite(r_std) or r_std < EPS:
        r_std = 1.0

    r_scaled = (z["sector_return"].to_numpy(dtype=float) - r_mean) / r_std
    feats = feature_matrix(r_scaled, feature_mode)

    # Positive target scale based on training only.
    train_target = z.loc[z["Date"] <= pd.Timestamp(train_end), target_col].dropna().values
    target_scale = np.nanmedian(train_target)
    if not np.isfinite(target_scale) or target_scale <= EPS:
        target_scale = np.nanmean(train_target)
    if not np.isfinite(target_scale) or target_scale <= EPS:
        target_scale = 1.0

    X, y, dates = [], [], []
    target = z[target_col].to_numpy(dtype=float)

    for i in range(lookback - 1, len(z)):
        if not np.isfinite(target[i]) or target[i] <= 0:
            continue
        xx = feats[i - lookback + 1:i + 1]
        if not np.isfinite(xx).all():
            continue
        X.append(xx)
        y.append(target[i] / target_scale)
        dates.append(z["Date"].iloc[i])

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    dates = pd.to_datetime(np.asarray(dates))

    train_mask = dates <= pd.Timestamp(train_end)
    valid_mask = (dates > pd.Timestamp(train_end)) & (dates <= pd.Timestamp(valid_end))
    test_mask = dates > pd.Timestamp(valid_end)

    return {
        "X_train": X[train_mask], "y_train": y[train_mask], "d_train": dates[train_mask],
        "X_valid": X[valid_mask], "y_valid": y[valid_mask], "d_valid": dates[valid_mask],
        "X_test": X[test_mask], "y_test": y[test_mask], "d_test": dates[test_mask],
        "target_scale": float(target_scale),
        "r_mean": float(r_mean), "r_std": float(r_std),
        "sector": sector, "h": h, "feature_mode": feature_mode,
        "target_proxy": target_proxy
    }

# ----------------------------------------------------------------------
# 8. LOSS FUNCTIONS AND METRICS
# ----------------------------------------------------------------------
@keras.utils.register_keras_serializable()
def qlike_tf(y_true, y_pred):
    y_true = tf.maximum(tf.cast(y_true, tf.float32), EPS)
    y_pred = tf.maximum(tf.cast(y_pred, tf.float32), EPS)
    ratio = y_true / y_pred
    return tf.reduce_mean(ratio - tf.math.log(ratio) - 1.0)

def qlike_np(y_true, y_pred):
    y_true = np.maximum(np.asarray(y_true, dtype=float), EPS)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), EPS)
    ratio = y_true / y_pred
    return ratio - np.log(ratio) - 1.0

def metric_dict(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "QLIKE": float(np.mean(qlike_np(y_true, y_pred))),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred))
    }

# ----------------------------------------------------------------------
# 9. MODEL BUILDERS
# ----------------------------------------------------------------------
def build_lstm(input_shape, cfg):
    x_in = layers.Input(shape=input_shape)
    x = layers.LSTM(cfg["units"], dropout=cfg.get("dropout", 0.0))(x_in)
    x = layers.Dense(cfg.get("dense", 16), activation="relu")(x)
    y = layers.Dense(1, activation="softplus")(x)
    model = Model(x_in, y, name="LSTM")
    return compile_model(model, cfg)

def build_gru(input_shape, cfg):
    x_in = layers.Input(shape=input_shape)
    x = layers.GRU(cfg["units"], dropout=cfg.get("dropout", 0.0))(x_in)
    x = layers.Dense(cfg.get("dense", 16), activation="relu")(x)
    y = layers.Dense(1, activation="softplus")(x)
    model = Model(x_in, y, name="GRU")
    return compile_model(model, cfg)

def tcn_residual_block(x, filters, kernel_size, dilation, dropout):
    shortcut = x
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters, kernel_size, padding="causal",
                      dilation_rate=dilation, activation="relu")(x)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same")(shortcut)
    return layers.Add()([x, shortcut])

def build_tcn(input_shape, cfg):
    x_in = layers.Input(shape=input_shape)
    x = x_in
    for dilation in cfg.get("dilations", [1, 2, 4]):
        x = tcn_residual_block(
            x, cfg["filters"], cfg.get("kernel_size", 3),
            dilation, cfg.get("dropout", 0.0)
        )
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(cfg.get("dense", 16), activation="relu")(x)
    y = layers.Dense(1, activation="softplus")(x)
    model = Model(x_in, y, name="TCN")
    return compile_model(model, cfg)

def transformer_encoder(x, d_model, n_heads, ff_dim, dropout):
    a = layers.MultiHeadAttention(
        num_heads=n_heads,
        key_dim=max(1, d_model // n_heads),
        dropout=dropout
    )(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(ff_dim, activation="gelu")(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(d_model)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    return x

def build_transformer(input_shape, cfg):
    x_in = layers.Input(shape=input_shape)
    d_model = cfg["d_model"]
    x = layers.Dense(d_model)(x_in)
    # Lightweight positional signal through learnable position embeddings
    positions = tf.range(start=0, limit=input_shape[0], delta=1)
    pos_emb = layers.Embedding(input_dim=input_shape[0], output_dim=d_model)(positions)
    x = x + pos_emb
    for _ in range(cfg.get("blocks", 2)):
        x = transformer_encoder(
            x, d_model, cfg["heads"], cfg["ff_dim"], cfg.get("dropout", 0.0)
        )
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(cfg.get("dense", 16), activation="relu")(x)
    y = layers.Dense(1, activation="softplus")(x)
    model = Model(x_in, y, name="TRANSFORMER")
    return compile_model(model, cfg)

def compile_model(model, cfg):
    opt = keras.optimizers.Adam(learning_rate=cfg.get("lr", 1e-3))
    model.compile(optimizer=opt, loss=qlike_tf)
    return model

MODEL_BUILDERS = {
    "LSTM": build_lstm,
    "GRU": build_gru,
    "TCN": build_tcn,
    "TRANSFORMER": build_transformer
}

# Equal tuning budget: same number of candidate configs per architecture.
CANDIDATES = {
    "LSTM": [
        {"units": 32, "dense": 16, "dropout": 0.10, "lr": 1e-3},
        {"units": 48, "dense": 16, "dropout": 0.10, "lr": 5e-4},
        {"units": 64, "dense": 32, "dropout": 0.15, "lr": 1e-3},
        {"units": 80, "dense": 32, "dropout": 0.20, "lr": 5e-4},
        {"units": 96, "dense": 32, "dropout": 0.20, "lr": 3e-4},
    ],
    "GRU": [
        {"units": 32, "dense": 16, "dropout": 0.10, "lr": 1e-3},
        {"units": 48, "dense": 16, "dropout": 0.10, "lr": 5e-4},
        {"units": 64, "dense": 32, "dropout": 0.15, "lr": 1e-3},
        {"units": 80, "dense": 32, "dropout": 0.20, "lr": 5e-4},
        {"units": 96, "dense": 32, "dropout": 0.20, "lr": 3e-4},
    ],
    "TCN": [
        {"filters": 16, "kernel_size": 3, "dilations": [1,2,4],
         "dense": 16, "dropout": 0.10, "lr": 1e-3},
        {"filters": 24, "kernel_size": 3, "dilations": [1,2,4],
         "dense": 16, "dropout": 0.10, "lr": 5e-4},
        {"filters": 32, "kernel_size": 3, "dilations": [1,2,4],
         "dense": 32, "dropout": 0.15, "lr": 1e-3},
        {"filters": 40, "kernel_size": 5, "dilations": [1,2,4],
         "dense": 32, "dropout": 0.20, "lr": 5e-4},
        {"filters": 48, "kernel_size": 3, "dilations": [1,2,4,8],
         "dense": 32, "dropout": 0.20, "lr": 3e-4},
    ],
    "TRANSFORMER": [
        {"d_model": 16, "heads": 2, "ff_dim": 32, "blocks": 1,
         "dense": 16, "dropout": 0.10, "lr": 1e-3},
        {"d_model": 24, "heads": 2, "ff_dim": 48, "blocks": 2,
         "dense": 16, "dropout": 0.10, "lr": 5e-4},
        {"d_model": 32, "heads": 4, "ff_dim": 64, "blocks": 2,
         "dense": 32, "dropout": 0.15, "lr": 1e-3},
        {"d_model": 48, "heads": 4, "ff_dim": 96, "blocks": 2,
         "dense": 32, "dropout": 0.20, "lr": 5e-4},
        {"d_model": 64, "heads": 4, "ff_dim": 128, "blocks": 2,
         "dense": 32, "dropout": 0.20, "lr": 3e-4},
    ],
}

# ----------------------------------------------------------------------
# 10. TRAINING / TUNING HELPERS
# ----------------------------------------------------------------------
def callbacks():
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, mode="min"
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=max(2, PATIENCE // 3), min_lr=1e-6
        )
    ]

def fit_one(model_name, cfg, ds, seed, use_train_plus_valid=False):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    builder = MODEL_BUILDERS[model_name]

    if use_train_plus_valid:
        Xtr = np.concatenate([ds["X_train"], ds["X_valid"]], axis=0)
        ytr = np.concatenate([ds["y_train"], ds["y_valid"]], axis=0)

        # Preserve a final tail from pre-test data for early stopping.
        n = len(Xtr)
        split = max(1, int(n * 0.90))
        X_fit, y_fit = Xtr[:split], ytr[:split]
        X_es, y_es = Xtr[split:], ytr[split:]
    else:
        X_fit, y_fit = ds["X_train"], ds["y_train"]
        X_es, y_es = ds["X_valid"], ds["y_valid"]

    if len(X_fit) < 50 or len(X_es) < 20:
        raise ValueError("Insufficient train/validation observations.")

    model = builder(X_fit.shape[1:], cfg)
    history = model.fit(
        X_fit, y_fit,
        validation_data=(X_es, y_es),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        verbose=VERBOSE_FIT,
        callbacks=callbacks()
    )
    return model, history

def tune_model(model_name, ds):
    candidates = CANDIDATES[model_name][:N_TUNING_CONFIGS]
    records = []

    for k, cfg in enumerate(candidates):
        model, hist = fit_one(model_name, cfg, ds, TUNING_SEED + k, False)
        pred = model.predict(ds["X_valid"], verbose=0).reshape(-1)
        score = float(np.mean(qlike_np(ds["y_valid"], pred)))
        records.append({
            "model": model_name, "candidate": k,
            "val_qlike": score, "params": model.count_params(),
            "config_json": json.dumps(cfg)
        })
        del model
        tf.keras.backend.clear_session()

    tab = pd.DataFrame(records).sort_values("val_qlike")
    best_cfg = json.loads(tab.iloc[0]["config_json"])
    return best_cfg, tab

def final_ensemble_forecast(model_name, cfg, ds, seeds):
    pred_list, seed_rows = [], []

    for seed in seeds:
        model, hist = fit_one(model_name, cfg, ds, seed, use_train_plus_valid=True)
        p_scaled = model.predict(ds["X_test"], verbose=0).reshape(-1)
        p = p_scaled * ds["target_scale"]
        y = ds["y_test"] * ds["target_scale"]
        pred_list.append(p)

        md = metric_dict(y, p)
        seed_rows.append({
            "seed": seed, "model": model_name, "sector": ds["sector"],
            "horizon": ds["h"], "target_proxy": ds["target_proxy"],
            "feature_mode": ds["feature_mode"], "params": model.count_params(),
            **md
        })
        del model
        tf.keras.backend.clear_session()

    P = np.vstack(pred_list)
    ensemble_pred = P.mean(axis=0)
    y_true = ds["y_test"] * ds["target_scale"]

    return y_true, ensemble_pred, pd.DataFrame(seed_rows)


# ----------------------------------------------------------------------
# 10B. CRASH-SAFE CHECKPOINT / RESUME HELPERS
# ----------------------------------------------------------------------
_COMPLETED_JOB_COUNTER = 0

def safe_slug(x):
    s = str(x).strip().replace("/", "_").replace("\\", "_")
    s = "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in s)
    while "__" in s:
        s = s.replace("__", "_")
    return s[:120]

def atomic_write_csv(df_, path_):
    path_ = Path(path_)
    path_.parent.mkdir(parents=True, exist_ok=True)
    tmp = path_.with_suffix(path_.suffix + ".tmp")
    df_.to_csv(tmp, index=False)
    os.replace(tmp, path_)

def atomic_write_json(obj_, path_):
    path_ = Path(path_)
    path_.parent.mkdir(parents=True, exist_ok=True)
    tmp = path_.with_suffix(path_.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj_, f, indent=2)
    os.replace(tmp, path_)

def make_progress_snapshot(force=False):
    global _COMPLETED_JOB_COUNTER
    if not ENABLE_PROGRESS_ZIP:
        return
    if (not force) and (_COMPLETED_JOB_COUNTER % SNAPSHOT_EVERY_N_COMPLETED_JOBS != 0):
        return
    base = ZIP_ROOT / f"progress_snapshot_{RUN_MODE}"
    try:
        shutil.make_archive(str(base), "zip", root_dir=OUTPUT_ROOT)
        print("    [backup] updated:", str(base) + ".zip")
    except Exception as e:
        print("    [backup warning]", e)

def checkpoint_job_dir(analysis_label, sector, model_name, horizon,
                       feature_mode="raw", target_proxy="rv", extra_key=""):
    parts = [
        safe_slug(analysis_label),
        f"h{horizon}",
        safe_slug(target_proxy),
        safe_slug(feature_mode),
        safe_slug(sector),
        safe_slug(model_name)
    ]
    if extra_key:
        parts.append(safe_slug(extra_key))
    d = CHECKPOINT_ROOT.joinpath(*parts)
    d.mkdir(parents=True, exist_ok=True)
    return d

def forecast_one_seed(model_name, cfg, ds, seed):
    model, _ = fit_one(model_name, cfg, ds, seed, use_train_plus_valid=True)
    p_scaled = model.predict(ds["X_test"], verbose=0).reshape(-1)
    p = p_scaled * ds["target_scale"]
    y = ds["y_test"] * ds["target_scale"]
    params = model.count_params()
    md = metric_dict(y, p)
    row = {
        "seed": seed,
        "model": model_name,
        "sector": ds["sector"],
        "horizon": ds["h"],
        "target_proxy": ds["target_proxy"],
        "feature_mode": ds["feature_mode"],
        "params": params,
        **md
    }
    del model
    tf.keras.backend.clear_session()
    return y, p, row

def load_seed_prediction(path_):
    z = pd.read_csv(path_, parse_dates=["Date"])
    return z["y_true"].to_numpy(float), z["y_pred"].to_numpy(float), pd.to_datetime(z["Date"])

def checkpointed_tuned_ensemble(model_name, ds, analysis_label, seeds,
                                retune=True, cfg_override=None, extra_key=""):
    """
    Crash-safe model job.

    - tuning result is saved immediately
    - each seed forecast is saved immediately
    - on restart, completed tuning/seeds are skipped
    - ensemble metrics/predictions are saved immediately
    """
    global _COMPLETED_JOB_COUNTER

    job_dir = checkpoint_job_dir(
        analysis_label, ds["sector"], model_name, ds["h"],
        ds["feature_mode"], ds["target_proxy"], extra_key
    )
    cfg_path = job_dir / "best_config.json"
    tuning_path = job_dir / "tuning.csv"
    complete_path = job_dir / "COMPLETE.json"
    ensemble_pred_path = job_dir / "ensemble_predictions.csv"
    ensemble_metric_path = job_dir / "ensemble_metrics.csv"

    # Configuration / tuning
    if cfg_override is not None:
        cfg = cfg_override
        if not cfg_path.exists():
            atomic_write_json(cfg, cfg_path)
        tuning_tab = pd.DataFrame()
    elif RESUME_COMPLETED_JOBS and cfg_path.exists() and tuning_path.exists():
        with open(cfg_path) as f:
            cfg = json.load(f)
        tuning_tab = pd.read_csv(tuning_path)
        print("    [resume] tuning/config loaded")
    else:
        cfg, tuning_tab = tune_model(model_name, ds)
        atomic_write_json(cfg, cfg_path)
        if not tuning_tab.empty:
            atomic_write_csv(tuning_tab, tuning_path)
        print("    [checkpoint] tuning saved")

    # Seed-level forecasts
    seed_rows = []
    pred_arrays = []
    y_ref = None
    d_ref = None

    for seed in seeds:
        pred_path = job_dir / f"seed_{seed}_predictions.csv"
        metric_path = job_dir / f"seed_{seed}_metrics.json"

        if RESUME_COMPLETED_JOBS and pred_path.exists() and metric_path.exists():
            y, p, d = load_seed_prediction(pred_path)
            with open(metric_path) as f:
                row = json.load(f)
            print(f"    [resume] seed {seed}")
        else:
            print(f"    fitting seed {seed} ...")
            y, p, row = forecast_one_seed(model_name, cfg, ds, seed)
            d = pd.to_datetime(ds["d_test"])
            seed_pred = pd.DataFrame({
                "Date": d,
                "y_true": y,
                "y_pred": p
            })
            atomic_write_csv(seed_pred, pred_path)
            atomic_write_json(row, metric_path)
            print(f"    [checkpoint] seed {seed} saved")

        if y_ref is None:
            y_ref = np.asarray(y, float)
            d_ref = pd.to_datetime(d)
        else:
            if len(y) != len(y_ref) or not np.allclose(np.asarray(y, float), y_ref, rtol=1e-7, atol=1e-12):
                raise RuntimeError(f"Seed checkpoint mismatch in {job_dir}")

        pred_arrays.append(np.asarray(p, float))
        seed_rows.append(row)

    ensemble_pred = np.mean(np.vstack(pred_arrays), axis=0)
    losses = qlike_np(y_ref, ensemble_pred)

    ensemble_df = pd.DataFrame({
        "Date": d_ref,
        "sector": ds["sector"],
        "model": model_name,
        "horizon": ds["h"],
        "target_proxy": ds["target_proxy"],
        "feature_mode": ds["feature_mode"],
        "y_true": y_ref,
        "y_pred": ensemble_pred,
        "qlike_loss": losses
    })
    ensemble_metric = {
        "sector": ds["sector"],
        "model": model_name,
        "horizon": ds["h"],
        "target_proxy": ds["target_proxy"],
        "feature_mode": ds["feature_mode"],
        "n_test": len(y_ref),
        **metric_dict(y_ref, ensemble_pred)
    }

    atomic_write_csv(ensemble_df, ensemble_pred_path)
    atomic_write_csv(pd.DataFrame([ensemble_metric]), ensemble_metric_path)
    atomic_write_json({
        "status": "complete",
        "analysis": analysis_label,
        "sector": ds["sector"],
        "model": model_name,
        "horizon": ds["h"],
        "seeds": list(seeds),
        "extra_key": extra_key,
        "completed_unix": time.time()
    }, complete_path)

    _COMPLETED_JOB_COUNTER += 1
    make_progress_snapshot(force=False)

    return (
        y_ref,
        ensemble_pred,
        pd.DataFrame(seed_rows),
        cfg,
        tuning_tab,
        ensemble_df
    )

def ensemble_from_saved_primary_seeds(sector, model_name, seeds):
    """
    Reuses h=5 primary seed forecasts without refitting.
    Used to build a horizon-robustness table with the same ROBUST_SEEDS
    as h=1 and h=22.
    """
    job_dir = checkpoint_job_dir(
        "primary", sector, model_name, PRIMARY_HORIZON, "raw", "rv", ""
    )
    preds = []
    y_ref = None
    d_ref = None
    for seed in seeds:
        pth = job_dir / f"seed_{seed}_predictions.csv"
        if not pth.exists():
            raise FileNotFoundError(
                f"Primary seed checkpoint missing: {pth}. "
                "Complete the primary h=5 run first."
            )
        y, p, d = load_seed_prediction(pth)
        if y_ref is None:
            y_ref, d_ref = y, d
        preds.append(p)
    pbar = np.mean(np.vstack(preds), axis=0)
    return y_ref, pbar, d_ref


def primary_subset_metrics_table(seeds):
    """Reconstruct primary h=5 metrics from any saved subset of primary seeds."""
    rows = []
    for sector in PRIMARY_SECTORS:
        for model_name in MODEL_NAMES:
            y, p, _ = ensemble_from_saved_primary_seeds(
                sector, model_name, seeds
            )
            rows.append({
                "sector": sector,
                "model": model_name,
                "weighting": "value",
                "horizon": PRIMARY_HORIZON,
                "n_test": len(y),
                **metric_dict(y, p)
            })
    return pd.DataFrame(rows)

# ----------------------------------------------------------------------
# 11. PRIMARY MANUSCRIPT RUN — CRASH-SAFE / RESUMABLE
# ----------------------------------------------------------------------
def run_primary(series, sectors, horizon, weighting_label="value"):
    all_metrics = []
    all_seed_metrics = []
    all_tuning = []
    all_predictions = []
    chosen_configs = {}

    main_metrics_path = OUTPUT_ROOT / "02_main_results" / "main_accuracy.csv"
    seed_metrics_path = OUTPUT_ROOT / "05_SI" / "random_seed_stability.csv"
    tuning_results_path = OUTPUT_ROOT / "05_SI" / "tuning_results.csv"
    main_predictions_path = OUTPUT_ROOT / "08_predictions" / "main_test_predictions.csv"
    config_path = OUTPUT_ROOT / "05_SI" / "chosen_hyperparameters.json"

    for sector in sectors:
        print(f"\nPRIMARY | {sector} | h={horizon}")
        ds = build_sequences_for_sector(
            series, sector, horizon, LOOKBACK, "raw", "rv"
        )

        if QUICK_TEST:
            # Keep only a small tail in smoke-test mode.
            # Dates must be trimmed together with X/y for test.
            for key in ["X_train", "y_train"]:
                ds[key] = ds[key][-500:]
            for key in ["X_valid", "y_valid"]:
                ds[key] = ds[key][-150:]
            for key in ["X_test", "y_test", "d_test"]:
                ds[key] = ds[key][-150:]

        for model_name in MODEL_NAMES:
            print("  ", model_name)
            seeds = MAIN_SEEDS if not QUICK_TEST else [11]

            y, p, seed_tab, cfg, tuning_tab, pred_df = checkpointed_tuned_ensemble(
                model_name=model_name,
                ds=ds,
                analysis_label="primary",
                seeds=seeds,
                retune=True,
                cfg_override=None,
                extra_key=""
            )

            if not tuning_tab.empty:
                tt = tuning_tab.copy()
                tt["sector"] = sector
                tt["horizon"] = horizon
                all_tuning.append(tt)

            chosen_configs[(sector, model_name)] = cfg
            all_seed_metrics.append(seed_tab)

            md = metric_dict(y, p)
            all_metrics.append({
                "sector": sector,
                "model": model_name,
                "weighting": weighting_label,
                "horizon": horizon,
                "n_test": len(y),
                **md
            })
            all_predictions.append(pred_df)

            # ----------------------------------------------------------
            # CRITICAL: persist aggregate results after EVERY model.
            # A Colab reset now loses at most the currently running seed.
            # ----------------------------------------------------------
            atomic_write_csv(pd.DataFrame(all_metrics), main_metrics_path)

            if all_seed_metrics:
                atomic_write_csv(
                    pd.concat(all_seed_metrics, ignore_index=True),
                    seed_metrics_path
                )

            if all_tuning:
                atomic_write_csv(
                    pd.concat(all_tuning, ignore_index=True),
                    tuning_results_path
                )

            if all_predictions:
                atomic_write_csv(
                    pd.concat(all_predictions, ignore_index=True),
                    main_predictions_path
                )

            atomic_write_json(
                {f"{k[0]}__{k[1]}": v for k, v in chosen_configs.items()},
                config_path
            )

    metrics = pd.DataFrame(all_metrics)
    seeds_df = pd.concat(all_seed_metrics, ignore_index=True)
    tuning_df = pd.concat(all_tuning, ignore_index=True) if all_tuning else pd.DataFrame()
    preds = pd.concat(all_predictions, ignore_index=True)

    # Final clean write after full loop.
    atomic_write_csv(metrics, main_metrics_path)
    atomic_write_csv(seeds_df, seed_metrics_path)
    if not tuning_df.empty:
        atomic_write_csv(tuning_df, tuning_results_path)
    atomic_write_csv(preds, main_predictions_path)
    atomic_write_json(
        {f"{k[0]}__{k[1]}": v for k, v in chosen_configs.items()},
        config_path
    )

    return metrics, seeds_df, tuning_df, preds, chosen_configs

main_metrics, seed_metrics, tuning_results, main_predictions, chosen_configs = run_primary(
    sector_value, PRIMARY_SECTORS, PRIMARY_HORIZON, "value"
)

# A forced progress snapshot after the complete primary run.
make_progress_snapshot(force=True)

# ----------------------------------------------------------------------
# 12. DIEBOLD-MARIANO TESTS WITH HAC STANDARD ERRORS
# ----------------------------------------------------------------------
def dm_test(loss_a, loss_b, horizon=1, hac_lags=None):
    a = np.asarray(loss_a, dtype=float)
    b = np.asarray(loss_b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    d = a[ok] - b[ok]

    if len(d) < 30:
        return np.nan, np.nan, len(d)

    X = np.ones((len(d), 1))
    lags = hac_lags
    if lags is None:
        lags = max(horizon - 1, int(np.floor(4 * (len(d) / 100.0) ** (2/9))))

    res = sm.OLS(d, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(lags)})
    stat = float(res.tvalues[0])
    p = float(res.pvalues[0])
    return stat, p, len(d)

def pairwise_dm_table(preds, horizon):
    rows = []
    for sector, g in preds.groupby("sector"):
        wide = g.pivot_table(index="Date", columns="model", values="qlike_loss")
        models = [m for m in MODEL_NAMES if m in wide.columns]
        for i in range(len(models)):
            for j in range(i + 1, len(models)):
                a, b = models[i], models[j]
                z = wide[[a, b]].dropna()
                stat, p, n = dm_test(z[a], z[b], horizon, DM_HAC_LAGS)
                rows.append({
                    "sector": sector, "model_A": a, "model_B": b,
                    "mean_loss_A": z[a].mean(), "mean_loss_B": z[b].mean(),
                    "DM_stat_A_minus_B": stat, "p_value": p, "n": n
                })
    return pd.DataFrame(rows)

dm_results = pairwise_dm_table(main_predictions, PRIMARY_HORIZON)
dm_results.to_csv(OUTPUT_ROOT / "02_main_results" / "DM_pairwise_QLIKE.csv", index=False)

# ----------------------------------------------------------------------
# 13. MODEL CONFIDENCE SET (RANGE-STATISTIC, BLOCK BOOTSTRAP)
# ----------------------------------------------------------------------
def moving_block_indices(T, block_len, rng):
    idx = []
    while len(idx) < T:
        start = rng.integers(0, T)
        block = [(start + k) % T for k in range(block_len)]
        idx.extend(block)
    return np.asarray(idx[:T])

def mcs_range(loss_df, alpha=0.10, B=1000, block_len=10, seed=2026):
    """
    Practical implementation of the Hansen-Lunde-Nason MCS elimination idea
    using the range statistic and moving-block bootstrap.

    Returns the surviving model set plus an elimination history.
    """
    L = loss_df.dropna().copy()
    active = list(L.columns)
    history = []
    rng = np.random.default_rng(seed)

    if len(active) <= 1 or len(L) < 30:
        return active, pd.DataFrame(history)

    while len(active) > 1:
        X = L[active].to_numpy(dtype=float)
        T, M = X.shape

        dbar = np.zeros((M, M))
        for i in range(M):
            for j in range(M):
                dbar[i, j] = np.mean(X[:, i] - X[:, j])

        boot_d = np.zeros((B, M, M))
        for b in range(B):
            idx = moving_block_indices(T, block_len, rng)
            Xb = X[idx]
            for i in range(M):
                for j in range(M):
                    boot_d[b, i, j] = np.mean(Xb[:, i] - Xb[:, j]) - dbar[i, j]

        se = np.std(boot_d, axis=0, ddof=1)
        se[se < EPS] = np.nan

        t_obs = np.abs(dbar / se)
        np.fill_diagonal(t_obs, np.nan)
        TR_obs = np.nanmax(t_obs)

        TR_boot = np.full(B, np.nan)
        for b in range(B):
            tb = np.abs(boot_d[b] / se)
            np.fill_diagonal(tb, np.nan)
            TR_boot[b] = np.nanmax(tb)

        pval = float(np.mean(TR_boot >= TR_obs))

        if pval >= alpha:
            history.append({
                "remaining": "|".join(active), "TR": TR_obs,
                "p_value": pval, "eliminated": ""
            })
            break

        # Eliminate model with largest average standardized loss differential.
        score = np.full(M, -np.inf)
        for i in range(M):
            vals = []
            for j in range(M):
                if i == j or not np.isfinite(se[i, j]):
                    continue
                vals.append(dbar[i, j] / se[i, j])
            score[i] = max(vals) if vals else -np.inf

        worst_idx = int(np.nanargmax(score))
        worst = active[worst_idx]
        history.append({
            "remaining": "|".join(active), "TR": TR_obs,
            "p_value": pval, "eliminated": worst
        })
        active.remove(worst)

    return active, pd.DataFrame(history)

mcs_rows = []
mcs_histories = []
for sector, g in main_predictions.groupby("sector"):
    wide = g.pivot_table(index="Date", columns="model", values="qlike_loss")
    wide = wide[[m for m in MODEL_NAMES if m in wide.columns]]
    survivors, hist = mcs_range(
        wide, MCS_ALPHA, MCS_BOOTSTRAPS, MCS_BLOCK_LENGTH, TUNING_SEED
    )
    mcs_rows.append({
        "sector": sector,
        "MCS_alpha": MCS_ALPHA,
        "surviving_models": "|".join(survivors),
        "n_survivors": len(survivors)
    })
    if not hist.empty:
        hist["sector"] = sector
        mcs_histories.append(hist)

mcs_summary = pd.DataFrame(mcs_rows)
mcs_summary.to_csv(OUTPUT_ROOT / "02_main_results" / "MCS_summary.csv", index=False)
if mcs_histories:
    pd.concat(mcs_histories, ignore_index=True).to_csv(
        OUTPUT_ROOT / "05_SI" / "MCS_elimination_history.csv", index=False
    )

# ----------------------------------------------------------------------
# 14. MODEL x SECTOR LOSS INTERACTION (DESCRIPTIVE/INFERENTIAL SUPPORT)
# ----------------------------------------------------------------------
# Fixed-effects style OLS with date-clustered SE is used as a supplementary
# interaction diagnostic. DM/MCS remain the principal forecast-comparison tests.
interaction_df = main_predictions[["Date", "sector", "model", "qlike_loss"]].dropna().copy()

try:
    formula = "qlike_loss ~ C(model) * C(sector)"
    interaction_fit = smf.ols(formula, data=interaction_df).fit(
        cov_type="cluster", cov_kwds={"groups": interaction_df["Date"]}
    )
    with open(OUTPUT_ROOT / "02_main_results" / "model_sector_interaction.txt", "w") as f:
        f.write(interaction_fit.summary().as_text())
except Exception as e:
    with open(OUTPUT_ROOT / "07_logs" / "interaction_model_error.txt", "w") as f:
        f.write(str(e))

# ----------------------------------------------------------------------
# 15. CAPACITY-MATCHED ARCHITECTURES
# ----------------------------------------------------------------------
def candidate_capacity_configs(model_name):
    if model_name == "LSTM":
        return [{"units": u, "dense": 16, "dropout": 0.10, "lr": 5e-4}
                for u in [24,32,40,48,56,64,72,80,96,112]]
    if model_name == "GRU":
        return [{"units": u, "dense": 16, "dropout": 0.10, "lr": 5e-4}
                for u in [24,32,40,48,56,64,72,80,96,112]]
    if model_name == "TCN":
        return [{"filters": f, "kernel_size": 3, "dilations": [1,2,4],
                 "dense": 16, "dropout": 0.10, "lr": 5e-4}
                for f in [8,12,16,20,24,28,32,40,48,56,64]]
    if model_name == "TRANSFORMER":
        out = []
        for d in [8,12,16,24,32,40,48,64]:
            for b in [1,2,3]:
                heads = 2 if d < 32 else 4
                out.append({"d_model": d, "heads": heads, "ff_dim": 2*d,
                            "blocks": b, "dense": 16,
                            "dropout": 0.10, "lr": 5e-4})
        return out
    raise ValueError(model_name)

def closest_capacity_config(model_name, input_shape, target=TARGET_PARAMETER_COUNT):
    rows = []
    for cfg in candidate_capacity_configs(model_name):
        tf.keras.backend.clear_session()
        m = MODEL_BUILDERS[model_name](input_shape, cfg)
        rows.append((abs(m.count_params() - target), m.count_params(), cfg))
        del m
    rows.sort(key=lambda x: x[0])
    return rows[0][2], rows[0][1]

if RUN_CAPACITY_MATCH:
    cap_metrics = []
    cap_specs = []
    cap_partial_path = OUTPUT_ROOT / "03_robustness" / "capacity_matched_accuracy.csv"
    cap_specs_path = OUTPUT_ROOT / "03_robustness" / "capacity_matched_specs.csv"

    for sector in PRIMARY_SECTORS:
        ds = build_sequences_for_sector(
            sector_value, sector, PRIMARY_HORIZON, LOOKBACK, "raw", "rv"
        )
        for model_name in MODEL_NAMES:
            print(f"CAPACITY | {sector} | {model_name} | h={PRIMARY_HORIZON}")
            cfg, n_params = closest_capacity_config(
                model_name, ds["X_train"].shape[1:], TARGET_PARAMETER_COUNT
            )
            y, p, _, _, _, _ = checkpointed_tuned_ensemble(
                model_name=model_name,
                ds=ds,
                analysis_label="capacity_matched",
                seeds=ROBUST_SEEDS if not QUICK_TEST else [11],
                retune=False,
                cfg_override=cfg,
                extra_key=f"target_{TARGET_PARAMETER_COUNT}"
            )
            cap_specs.append({
                "sector": sector,
                "model": model_name,
                "parameter_target": TARGET_PARAMETER_COUNT,
                "actual_params": n_params,
                "config_json": json.dumps(cfg)
            })
            cap_metrics.append({
                "sector": sector,
                "model": model_name,
                "horizon": PRIMARY_HORIZON,
                **metric_dict(y, p)
            })

            atomic_write_csv(pd.DataFrame(cap_metrics), cap_partial_path)
            atomic_write_csv(pd.DataFrame(cap_specs), cap_specs_path)

    atomic_write_csv(pd.DataFrame(cap_metrics), cap_partial_path)
    atomic_write_csv(pd.DataFrame(cap_specs), cap_specs_path)

# ----------------------------------------------------------------------
# 16. HORIZON ROBUSTNESS
# ----------------------------------------------------------------------
def run_reduced_robustness(series, sectors, horizons, label, target_proxy="rv",
                           feature_mode="raw", seeds=None, extra_key=""):
    """
    Crash-safe robustness runner.
    Each sector x model x horizon is checkpointed independently.
    """
    if seeds is None:
        seeds = ROBUST_SEEDS

    rows = []
    partial_path = OUTPUT_ROOT / "03_robustness" / f"{safe_slug(label)}_partial.csv"

    for h in horizons:
        for sector in sectors:
            ds = build_sequences_for_sector(
                series, sector, h, LOOKBACK, feature_mode, target_proxy
            )
            for model_name in MODEL_NAMES:
                print(f"{label.upper()} | {sector} | {model_name} | h={h}")
                y, p, _, _, _, _ = checkpointed_tuned_ensemble(
                    model_name=model_name,
                    ds=ds,
                    analysis_label=label,
                    seeds=seeds if not QUICK_TEST else [11],
                    retune=True,
                    cfg_override=None,
                    extra_key=extra_key
                )
                rows.append({
                    "analysis": label,
                    "sector": sector,
                    "model": model_name,
                    "horizon": h,
                    "feature_mode": feature_mode,
                    "target_proxy": target_proxy,
                    **metric_dict(y, p)
                })
                atomic_write_csv(pd.DataFrame(rows), partial_path)

    return pd.DataFrame(rows)

if RUN_HORIZON_ROBUSTNESS:
    # --------------------------------------------------------------
    # IMPORTANT: only h=1 and h=22 are newly estimated.
    # h=5 is REUSED from the saved primary seed forecasts.
    # No h=5 retraining occurs here.
    # --------------------------------------------------------------
    horizon_new = run_reduced_robustness(
        sector_value,
        PRIMARY_SECTORS,
        ROBUST_HORIZONS,   # [1, 22], not [1, 5, 22]
        "horizon_robustness",
        "rv",
        "raw",
        ROBUST_SEEDS if not QUICK_TEST else [11],
        extra_key="additional_horizons"
    )

    # Rebuild h=5 using the same ROBUST_SEEDS subset already produced in
    # the primary run, so the horizon comparison uses a consistent seed set.
    h5_rows = []
    h5_seeds = ROBUST_SEEDS if not QUICK_TEST else [11]
    for sector in PRIMARY_SECTORS:
        for model_name in MODEL_NAMES:
            y5, p5, _ = ensemble_from_saved_primary_seeds(
                sector, model_name, h5_seeds
            )
            h5_rows.append({
                "analysis": "horizon_robustness_reused_primary",
                "sector": sector,
                "model": model_name,
                "horizon": PRIMARY_HORIZON,
                "feature_mode": "raw",
                "target_proxy": "rv",
                **metric_dict(y5, p5)
            })

    horizon_h5 = pd.DataFrame(h5_rows)
    horizon_metrics = pd.concat(
        [horizon_new, horizon_h5], ignore_index=True
    ).sort_values(["horizon", "sector", "model"])

    atomic_write_csv(
        horizon_new,
        OUTPUT_ROOT / "03_robustness" / "horizon_robustness_new_fits_only.csv"
    )
    atomic_write_csv(
        horizon_metrics,
        OUTPUT_ROOT / "03_robustness" / "horizon_robustness_all_horizons.csv"
    )

# ----------------------------------------------------------------------
# 17. VOLATILITY-REGIME ROBUSTNESS USING MAIN OOS PREDICTIONS
# ----------------------------------------------------------------------
if RUN_REGIME_ROBUSTNESS:
    regime_rows = []
    for sector in PRIMARY_SECTORS:
        # Thresholds are estimated from pre-test realized variance only.
        ds_pre = build_sequences_for_sector(
            sector_value, sector, PRIMARY_HORIZON, LOOKBACK, "raw", "rv"
        )
        pre_y = np.concatenate([ds_pre["y_train"], ds_pre["y_valid"]]) * ds_pre["target_scale"]
        q33, q67 = np.quantile(pre_y[np.isfinite(pre_y)], [1/3, 2/3])

        g = main_predictions[main_predictions["sector"] == sector].copy()
        g["regime"] = np.where(
            g["y_true"] <= q33, "Low",
            np.where(g["y_true"] <= q67, "Normal", "High")
        )

        for (model_name, regime), x in g.groupby(["model", "regime"]):
            regime_rows.append({
                "sector": sector, "model": model_name, "regime": regime,
                "n": len(x),
                "QLIKE": x["qlike_loss"].mean(),
                "RMSE": np.sqrt(np.mean((x["y_true"] - x["y_pred"])**2)),
                "MAE": np.mean(np.abs(x["y_true"] - x["y_pred"]))
            })

    pd.DataFrame(regime_rows).to_csv(
        OUTPUT_ROOT / "03_robustness" / "volatility_regime_accuracy.csv", index=False
    )

# ----------------------------------------------------------------------
# 18. SIGNED-RETURN / ASYMMETRY ABLATION
# ----------------------------------------------------------------------
if RUN_SIGNED_RETURN_ABLATION:
    ablation_rows = []
    ablation_path = OUTPUT_ROOT / "04_ablations" / "signed_return_ablation.csv"

    for sector in PRIMARY_SECTORS:
        for feature_mode in ["posneg", "positive_only", "negative_only"]:
            ds = build_sequences_for_sector(
                sector_value, sector, PRIMARY_HORIZON, LOOKBACK,
                feature_mode, "rv"
            )
            for model_name in MODEL_NAMES:
                print(
                    f"ABLATION | {feature_mode} | {sector} | "
                    f"{model_name} | h={PRIMARY_HORIZON}"
                )
                y, p, _, _, _, _ = checkpointed_tuned_ensemble(
                    model_name=model_name,
                    ds=ds,
                    analysis_label=f"signed_return_ablation_{feature_mode}",
                    seeds=ROBUST_SEEDS if not QUICK_TEST else [11],
                    retune=True,
                    cfg_override=None,
                    extra_key=""
                )
                ablation_rows.append({
                    "sector": sector,
                    "model": model_name,
                    "feature_mode": feature_mode,
                    "horizon": PRIMARY_HORIZON,
                    **metric_dict(y, p)
                })
                atomic_write_csv(pd.DataFrame(ablation_rows), ablation_path)

    ablation_tab = pd.DataFrame(ablation_rows)
    atomic_write_csv(ablation_tab, ablation_path)

    # Incremental QLIKE loss relative to full positive+negative channels.
    full = ablation_tab[ablation_tab["feature_mode"] == "posneg"][
        ["sector", "model", "QLIKE"]
    ].rename(columns={"QLIKE": "QLIKE_full"})
    delta = ablation_tab.merge(full, on=["sector", "model"], how="left")
    delta["Delta_QLIKE_vs_full"] = delta["QLIKE"] - delta["QLIKE_full"]
    atomic_write_csv(
        delta,
        OUTPUT_ROOT / "04_ablations" / "signed_return_ablation_deltas.csv"
    )

# ----------------------------------------------------------------------
# 19. EQUAL-WEIGHTED SECTOR ROBUSTNESS (SI)
# ----------------------------------------------------------------------
if RUN_EQUAL_WEIGHT_SI:
    ew = run_reduced_robustness(
        sector_equal, PRIMARY_SECTORS, [PRIMARY_HORIZON],
        "equal_weighted_sector", "rv", "raw",
        ROBUST_SEEDS if not QUICK_TEST else [11]
    )
    ew.to_csv(
        OUTPUT_ROOT / "05_SI" / "equal_weighted_sector_accuracy.csv", index=False
    )

    # Compare value-weighted vs equal-weighted rankings.
    vw_rank = primary_subset_metrics_table(
        ROBUST_SEEDS if not QUICK_TEST else [11]
    )
    vw_rank["rank"] = vw_rank.groupby("sector")["QLIKE"].rank(method="min")
    ew_rank = ew.copy()
    ew_rank["rank"] = ew_rank.groupby("sector")["QLIKE"].rank(method="min")
    rank_compare = vw_rank[["sector", "model", "rank"]].merge(
        ew_rank[["sector", "model", "rank"]],
        on=["sector", "model"], suffixes=("_value", "_equal")
    )
    rank_compare.to_csv(
        OUTPUT_ROOT / "05_SI" / "value_vs_equal_rankings.csv", index=False
    )

# ----------------------------------------------------------------------
# 20. ALTERNATIVE PARKINSON-BASED VOLATILITY TARGET (SI)
# ----------------------------------------------------------------------
if RUN_PARKINSON_PROXY_SI:
    pk = run_reduced_robustness(
        sector_value, PRIMARY_SECTORS, [PRIMARY_HORIZON],
        "parkinson_constituent_proxy", "parkinson", "raw",
        ROBUST_SEEDS if not QUICK_TEST else [11]
    )
    pk.to_csv(
        OUTPUT_ROOT / "05_SI" / "parkinson_proxy_accuracy.csv", index=False
    )

# ----------------------------------------------------------------------
# 21. THIN-TRADING ROBUSTNESS (SI)
# ----------------------------------------------------------------------
if RUN_THIN_TRADING_SI:
    eligible_stocks = stock_thin.loc[
        stock_thin["zero_return_rate"] <= THIN_TRADING_MAX_ZERO_RATE, "Stock"
    ].tolist()

    thin_panel = panel[panel["Stock"].isin(eligible_stocks)].copy()
    thin_sector_counts = (
        thin_panel.groupby("Category")["Stock"].nunique().rename("n").reset_index()
    )
    thin_primary = thin_sector_counts.loc[
        thin_sector_counts["n"] >= PRIMARY_MIN_STOCKS, "Category"
    ].tolist()

    thin_series = build_sector_series(thin_panel, "value", thin_primary)
    thin_results = run_reduced_robustness(
        thin_series, thin_primary, [PRIMARY_HORIZON],
        "thin_trading_filtered", "rv", "raw",
        ROBUST_SEEDS if not QUICK_TEST else [11]
    )
    thin_results.to_csv(
        OUTPUT_ROOT / "05_SI" / "thin_trading_filtered_accuracy.csv", index=False
    )
    thin_sector_counts.to_csv(
        OUTPUT_ROOT / "05_SI" / "thin_trading_filtered_sector_counts.csv", index=False
    )

# ----------------------------------------------------------------------
# 22. EXTREME-RETURN SENSITIVITY (SI)
# ----------------------------------------------------------------------
def winsorize_sector_returns(sector_series, q=0.005):
    z = sector_series.copy()
    out = []
    for sector, g in z.groupby("Category"):
        g = g.copy()
        # Limits determined only from the pre-test sample.
        ref = g.loc[g["Date"] <= pd.Timestamp(VALID_END), "sector_return"].dropna()
        lo, hi = ref.quantile([q, 1-q])
        g["sector_return"] = g["sector_return"].clip(lo, hi)
        out.append(g)
    return pd.concat(out, ignore_index=True)

if RUN_EXTREME_RETURN_SI:
    wins_series = winsorize_sector_returns(sector_value, EXTREME_RETURN_WINSOR_Q)
    wins = run_reduced_robustness(
        wins_series, PRIMARY_SECTORS, [PRIMARY_HORIZON],
        "winsorized_return_sensitivity", "rv", "raw",
        ROBUST_SEEDS if not QUICK_TEST else [11]
    )
    wins.to_csv(
        OUTPUT_ROOT / "05_SI" / "extreme_return_winsorized_accuracy.csv", index=False
    )

# ----------------------------------------------------------------------
# 23. LEARNING-CURVE ROBUSTNESS (SI)
# ----------------------------------------------------------------------
if RUN_LEARNING_CURVES_SI:
    lc_rows = []
    lc_path = OUTPUT_ROOT / "05_SI" / "learning_curves.csv"

    # To control computing cost, use already selected primary configs.
    for sector in PRIMARY_SECTORS:
        ds_full = build_sequences_for_sector(
            sector_value, sector, PRIMARY_HORIZON, LOOKBACK, "raw", "rv"
        )

        for frac in LEARNING_CURVE_FRACTIONS:
            n = len(ds_full["X_train"])
            start = int(n * (1.0 - frac))
            ds = dict(ds_full)
            ds["X_train"] = ds_full["X_train"][start:]
            ds["y_train"] = ds_full["y_train"][start:]
            ds["d_train"] = ds_full["d_train"][start:]

            for model_name in MODEL_NAMES:
                print(
                    f"LEARNING CURVE | frac={frac:.2f} | {sector} | "
                    f"{model_name}"
                )
                cfg = chosen_configs[(sector, model_name)]
                y, p, _, _, _, _ = checkpointed_tuned_ensemble(
                    model_name=model_name,
                    ds=ds,
                    analysis_label="learning_curve",
                    seeds=[ROBUST_SEEDS[0]] if not QUICK_TEST else [11],
                    retune=False,
                    cfg_override=cfg,
                    extra_key=f"trainfrac_{frac:.2f}"
                )
                lc_rows.append({
                    "sector": sector,
                    "model": model_name,
                    "train_fraction": frac,
                    "n_train": len(ds["X_train"]),
                    **metric_dict(y, p)
                })
                atomic_write_csv(pd.DataFrame(lc_rows), lc_path)

    atomic_write_csv(pd.DataFrame(lc_rows), lc_path)

# ----------------------------------------------------------------------
# 24. LOCAL VS GLOBAL TRAINING ROBUSTNESS (SI)
# ----------------------------------------------------------------------
def append_sector_onehot(X, sector_index, n_sectors):
    n, t, _ = X.shape
    one = np.zeros((n, t, n_sectors), dtype=np.float32)
    one[:, :, sector_index] = 1.0
    return np.concatenate([X, one], axis=2)

def build_global_dataset(series, sectors, h, lookback=LOOKBACK):
    dsets = {}
    for s in sectors:
        dsets[s] = build_sequences_for_sector(
            series, s, h, lookback, "raw", "rv"
        )

    global_ds = {}
    for split in ["train", "valid", "test"]:
        Xs, ys, sectors_meta, dates_meta, scales = [], [], [], [], []
        for si, s in enumerate(sectors):
            ds = dsets[s]
            X = append_sector_onehot(ds[f"X_{split}"], si, len(sectors))
            y = ds[f"y_{split}"]  # sector-normalized target
            Xs.append(X)
            ys.append(y)
            sectors_meta.extend([s] * len(y))
            dates_meta.extend(ds[f"d_{split}"])
            scales.extend([ds["target_scale"]] * len(y))
        global_ds[f"X_{split}"] = np.concatenate(Xs)
        global_ds[f"y_{split}"] = np.concatenate(ys)
        global_ds[f"sector_{split}"] = np.array(sectors_meta)
        global_ds[f"date_{split}"] = pd.to_datetime(np.array(dates_meta))
        global_ds[f"scale_{split}"] = np.array(scales, dtype=float)

    return global_ds

def fit_global_model(model_name, cfg, gds, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()
    model = MODEL_BUILDERS[model_name](gds["X_train"].shape[1:], cfg)
    model.fit(
        gds["X_train"], gds["y_train"],
        validation_data=(gds["X_valid"], gds["y_valid"]),
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        shuffle=False, verbose=VERBOSE_FIT, callbacks=callbacks()
    )
    pred_scaled = model.predict(gds["X_test"], verbose=0).reshape(-1)
    y = gds["y_test"] * gds["scale_test"]
    p = pred_scaled * gds["scale_test"]
    return model, y, p

if RUN_LOCAL_GLOBAL_SI:
    gds = build_global_dataset(
        sector_value, PRIMARY_SECTORS, PRIMARY_HORIZON, LOOKBACK
    )
    global_rows = []
    lg_output = OUTPUT_ROOT / "05_SI" / "local_vs_global_accuracy.csv"

    for model_name in MODEL_NAMES:
        print(f"GLOBAL TRAINING | {model_name} | h={PRIMARY_HORIZON}")
        gjob = CHECKPOINT_ROOT / "local_vs_global" / safe_slug(model_name)
        gjob.mkdir(parents=True, exist_ok=True)
        metrics_path = gjob / "global_sector_metrics.csv"
        cfg_path = gjob / "best_global_config.json"
        complete_path = gjob / "COMPLETE.json"

        if (
            RESUME_COMPLETED_JOBS
            and complete_path.exists()
            and metrics_path.exists()
        ):
            print("    [resume] completed global model loaded")
            model_sector_rows = pd.read_csv(metrics_path).to_dict("records")
        else:
            # ---------------- global tuning ----------------
            if RESUME_COMPLETED_JOBS and cfg_path.exists():
                with open(cfg_path) as f:
                    best_global_cfg = json.load(f)
                print("    [resume] global tuning/config loaded")
            else:
                candidate_records = []
                for k, cfg in enumerate(CANDIDATES[model_name][:N_TUNING_CONFIGS]):
                    set_all_seeds(TUNING_SEED + k)
                    tf.keras.backend.clear_session()
                    m = MODEL_BUILDERS[model_name](
                        gds["X_train"].shape[1:], cfg
                    )
                    m.fit(
                        gds["X_train"], gds["y_train"],
                        validation_data=(gds["X_valid"], gds["y_valid"]),
                        epochs=MAX_EPOCHS,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        verbose=VERBOSE_FIT,
                        callbacks=callbacks()
                    )
                    pv = m.predict(gds["X_valid"], verbose=0).reshape(-1)
                    score = np.mean(qlike_np(gds["y_valid"], pv))
                    candidate_records.append({
                        "candidate": k,
                        "val_qlike": float(score),
                        "config_json": json.dumps(cfg),
                        "params": int(m.count_params())
                    })
                    del m
                    tf.keras.backend.clear_session()

                cand_df = pd.DataFrame(candidate_records).sort_values("val_qlike")
                best_global_cfg = json.loads(cand_df.iloc[0]["config_json"])
                atomic_write_csv(cand_df, gjob / "global_tuning.csv")
                atomic_write_json(best_global_cfg, cfg_path)
                print("    [checkpoint] global tuning saved")

            # ---------------- global seed fits ----------------
            all_seed_pred = []
            y_ref = None
            global_seeds = ROBUST_SEEDS if not QUICK_TEST else [11]

            for seed in global_seeds:
                sp = gjob / f"seed_{seed}_predictions.csv"
                if RESUME_COMPLETED_JOBS and sp.exists():
                    zz = pd.read_csv(sp)
                    y = zz["y_true"].to_numpy(float)
                    p = zz["y_pred"].to_numpy(float)
                    print(f"    [resume] global seed {seed}")
                else:
                    print(f"    fitting global seed {seed} ...")
                    m, y, p = fit_global_model(
                        model_name, best_global_cfg, gds, seed
                    )
                    zz = pd.DataFrame({
                        "Date": gds["date_test"],
                        "sector": gds["sector_test"],
                        "y_true": y,
                        "y_pred": p
                    })
                    atomic_write_csv(zz, sp)
                    del m
                    tf.keras.backend.clear_session()
                    print(f"    [checkpoint] global seed {seed} saved")

                y_ref = y if y_ref is None else y_ref
                all_seed_pred.append(p)

            pg = np.mean(np.vstack(all_seed_pred), axis=0)
            model_sector_rows = []
            for sector in PRIMARY_SECTORS:
                mask = gds["sector_test"] == sector
                md = metric_dict(y_ref[mask], pg[mask])
                model_sector_rows.append({
                    "training_scope": "global",
                    "sector": sector,
                    "model": model_name,
                    **md
                })

            atomic_write_csv(pd.DataFrame(model_sector_rows), metrics_path)
            atomic_write_json({
                "status": "complete",
                "model": model_name,
                "seeds": list(global_seeds),
                "completed_unix": time.time()
            }, complete_path)

        global_rows.extend(model_sector_rows)

        # Persist partial local-vs-global table after every completed global model.
        global_partial = pd.DataFrame(global_rows)
        local_tab_partial = primary_subset_metrics_table(
            ROBUST_SEEDS if not QUICK_TEST else [11]
        )
        local_tab_partial["training_scope"] = "local"
        local_tab_partial = local_tab_partial[
            ["training_scope", "sector", "model", "QLIKE", "RMSE", "MAE"]
        ]
        atomic_write_csv(
            pd.concat([local_tab_partial, global_partial], ignore_index=True),
            lg_output
        )

    global_tab = pd.DataFrame(global_rows)
    local_tab = primary_subset_metrics_table(
        ROBUST_SEEDS if not QUICK_TEST else [11]
    )
    local_tab["training_scope"] = "local"
    local_tab = local_tab[
        ["training_scope", "sector", "model", "QLIKE", "RMSE", "MAE"]
    ]
    atomic_write_csv(
        pd.concat([local_tab, global_tab], ignore_index=True),
        lg_output
    )

# ----------------------------------------------------------------------
# 25. OPTIONAL LOOKBACK ABLATION (HEAVY SI)
# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# 25. OPTIONAL LOOKBACK ABLATION (HEAVY SI)
# ----------------------------------------------------------------------
if RUN_LOOKBACK_ABLATION_SI:
    rows = []
    for lb in LOOKBACK_GRID:
        for sector in PRIMARY_SECTORS:
            ds = build_sequences_for_sector(
                sector_value, sector, PRIMARY_HORIZON, lb, "raw", "rv"
            )
            for model_name in MODEL_NAMES:
                cfg, _ = tune_model(model_name, ds)
                y, p, _ = final_ensemble_forecast(
                    model_name, cfg, ds, [ROBUST_SEEDS[0]]
                )
                rows.append({
                    "sector": sector, "model": model_name,
                    "lookback": lb, **metric_dict(y, p)
                })
    pd.DataFrame(rows).to_csv(
        OUTPUT_ROOT / "05_SI" / "lookback_ablation.csv", index=False
    )

# ----------------------------------------------------------------------
# 26. OPTIONAL EXPANDING-WINDOW OOS ROBUSTNESS (HEAVY SI)
# ----------------------------------------------------------------------
# This module is intentionally conservative: it re-estimates at quarterly
# blocks with the primary selected hyperparameters. It is OFF by default
# because it is computationally expensive.
def expanding_oos_forecast_one(series, sector, model_name, cfg, h,
                               refit_every=63, seed=11):
    ds_full = build_sequences_for_sector(
        series, sector, h, LOOKBACK, "raw", "rv"
    )

    # Build one chronological combined sequence using the same scaling scheme.
    X_all = np.concatenate([ds_full["X_train"], ds_full["X_valid"], ds_full["X_test"]])
    y_all = np.concatenate([ds_full["y_train"], ds_full["y_valid"], ds_full["y_test"]])
    d_all = np.concatenate([ds_full["d_train"], ds_full["d_valid"], ds_full["d_test"]])

    test_idx = np.where(pd.to_datetime(d_all) > pd.Timestamp(VALID_END))[0]
    preds, truths, dates = [], [], []

    for b_start in range(0, len(test_idx), refit_every):
        block_idx = test_idx[b_start:b_start + refit_every]
        first_test_position = block_idx[0]
        train_positions = np.arange(0, first_test_position)

        # Tail pre-test validation block for early stopping.
        split = max(50, int(len(train_positions) * 0.90))
        tr = train_positions[:split]
        va = train_positions[split:]
        if len(va) < 20:
            continue

        set_all_seeds(seed)
        tf.keras.backend.clear_session()
        m = MODEL_BUILDERS[model_name](X_all.shape[1:], cfg)
        m.fit(
            X_all[tr], y_all[tr],
            validation_data=(X_all[va], y_all[va]),
            epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
            shuffle=False, verbose=VERBOSE_FIT, callbacks=callbacks()
        )
        p = m.predict(X_all[block_idx], verbose=0).reshape(-1)
        preds.extend(p * ds_full["target_scale"])
        truths.extend(y_all[block_idx] * ds_full["target_scale"])
        dates.extend(pd.to_datetime(d_all[block_idx]))
        del m
        tf.keras.backend.clear_session()

    return np.array(truths), np.array(preds), pd.to_datetime(dates)

if RUN_EXPANDING_WINDOW_SI:
    rows = []
    for sector in PRIMARY_SECTORS:
        for model_name in MODEL_NAMES:
            cfg = chosen_configs[(sector, model_name)]
            y, p, d = expanding_oos_forecast_one(
                sector_value, sector, model_name, cfg,
                PRIMARY_HORIZON, 63, ROBUST_SEEDS[0]
            )
            rows.append({
                "sector": sector, "model": model_name,
                "refit_every_days": 63, **metric_dict(y, p)
            })
    pd.DataFrame(rows).to_csv(
        OUTPUT_ROOT / "05_SI" / "expanding_window_accuracy.csv", index=False
    )

# ----------------------------------------------------------------------
# 27. OPTIONAL STOCK-LEVEL ROBUSTNESS (HEAVY SI)
# ----------------------------------------------------------------------
# Kept OFF by default because it can multiply the number of fits by ~52.
# When enabled, it forecasts stock volatility separately then summarizes
# accuracy within sectors.
def stock_sequence_dataset(panel_df, stock, h, lookback=LOOKBACK):
    g = panel_df[panel_df["Stock"] == stock].copy().sort_values("Date")
    g["sq_return"] = g["log_return"] ** 2
    g[f"rv_h{h}"] = future_sum(g["sq_return"], h)
    g = g.rename(columns={"log_return": "sector_return"})
    g["Category"] = g["Category"].ffill().bfill()
    g["sector_parkinson_var"] = g["parkinson_var"]
    return build_sequences_for_sector(
        g.rename(columns={"Category": "Category"}),
        g["Category"].dropna().iloc[0], h, lookback, "raw", "rv"
    )

if RUN_STOCK_LEVEL_SI:
    stock_rows = []
    for sector in PRIMARY_SECTORS:
        stocks = stock_coverage.loc[stock_coverage["Category"] == sector, "Stock"].tolist()
        for stock in stocks:
            # Construct a one-stock "sector" dataframe in expected format.
            g = panel[panel["Stock"] == stock][
                ["Date", "Category", "log_return", "parkinson_var"]
            ].copy()
            g = g.rename(columns={
                "log_return": "sector_return",
                "parkinson_var": "sector_parkinson_var"
            })
            if g["sector_return"].notna().sum() < 500:
                continue
            ds = build_sequences_for_sector(
                g, sector, PRIMARY_HORIZON, LOOKBACK, "raw", "rv"
            )
            for model_name in MODEL_NAMES:
                cfg, _ = tune_model(model_name, ds)
                y, p, _ = final_ensemble_forecast(
                    model_name, cfg, ds, [ROBUST_SEEDS[0]]
                )
                stock_rows.append({
                    "sector": sector, "stock": stock, "model": model_name,
                    **metric_dict(y, p)
                })
    pd.DataFrame(stock_rows).to_csv(
        OUTPUT_ROOT / "05_SI" / "stock_level_accuracy.csv", index=False
    )

# ----------------------------------------------------------------------
# 28. MAIN FIGURES
# ----------------------------------------------------------------------
# Figure 1: relative QLIKE heatmap (model x sector)
pivot = main_metrics.pivot(index="sector", columns="model", values="QLIKE")
rel = pivot.div(pivot.min(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(rel.values, aspect="auto")
ax.set_xticks(np.arange(len(rel.columns)))
ax.set_xticklabels(rel.columns, rotation=30, ha="right")
ax.set_yticks(np.arange(len(rel.index)))
ax.set_yticklabels(rel.index)
ax.set_title("Relative Out-of-Sample QLIKE by Architecture and Sector")
fig.colorbar(im, ax=ax, label="QLIKE / sector-best QLIKE")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "06_figures" / "Figure_relative_QLIKE_heatmap.png", dpi=300)
plt.close(fig)

# Figure 2: average architecture QLIKE
avg = main_metrics.groupby("model")["QLIKE"].mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(avg.index, avg.values)
ax.set_ylabel("Mean OOS QLIKE")
ax.set_title("Average Out-of-Sample Volatility Forecast Loss")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "06_figures" / "Figure_average_QLIKE.png", dpi=300)
plt.close(fig)

# Figure 3: sector concentration
fig, ax = plt.subplots(figsize=(8, 5))
c = concentration.set_index("Category")["mean_largest_weight"].sort_values()
ax.barh(c.index, c.values)
ax.set_xlabel("Mean largest lagged market-cap weight")
ax.set_title("Sector Portfolio Concentration")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "06_figures" / "Figure_sector_concentration.png", dpi=300)
plt.close(fig)

# ----------------------------------------------------------------------
# 29. MANUSCRIPT-READY SUMMARY TABLES
# ----------------------------------------------------------------------
# Best model per sector
best_by_sector = (
    main_metrics.sort_values(["sector", "QLIKE"])
                .groupby("sector", as_index=False).first()
)
best_by_sector.to_csv(
    OUTPUT_ROOT / "02_main_results" / "best_model_by_sector.csv", index=False
)

# Relative loss to sector-best
relative_loss = main_metrics.copy()
relative_loss["sector_best_QLIKE"] = relative_loss.groupby("sector")["QLIKE"].transform("min")
relative_loss["relative_QLIKE"] = relative_loss["QLIKE"] / relative_loss["sector_best_QLIKE"]
relative_loss["percent_excess_QLIKE"] = 100 * (relative_loss["relative_QLIKE"] - 1)
relative_loss.to_csv(
    OUTPUT_ROOT / "02_main_results" / "relative_loss_by_sector.csv", index=False
)

# ----------------------------------------------------------------------
# 30. RUN MANIFEST
# ----------------------------------------------------------------------
manifest = {
    "title": "Does Architecture Matter for Volatility Forecasting? Evidence Across Heterogeneous Equity Sectors",
    "hypothesis": "H02: There is no statistically significant difference in the out-of-sample volatility forecast accuracy among selected deep-learning models across Nairobi Securities Exchange sectors.",
    "data_file": str(DATA_PATH),
    "data_rows": int(len(df)),
    "date_min": str(df["Date"].min().date()),
    "date_max": str(df["Date"].max().date()),
    "n_stocks": int(df["Stock"].nunique()),
    "n_sectors": int(df["Category"].nunique()),
    "primary_sectors": PRIMARY_SECTORS,
    "primary_weighting": PRIMARY_WEIGHTING,
    "lagged_mcap_weights": True,
    "primary_horizon": PRIMARY_HORIZON,
    "robust_horizons_new_fits_only": ROBUST_HORIZONS,
    "report_horizons": REPORT_HORIZONS,
    "h5_reused_from_primary": True,
    "persistent_project_root": str(PROJECT_ROOT),
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "lookback": LOOKBACK,
    "train_end": TRAIN_END,
    "valid_end": VALID_END,
    "test_start": str((pd.Timestamp(VALID_END) + pd.Timedelta(days=1)).date()),
    "models": MODEL_NAMES,
    "main_seeds": MAIN_SEEDS if not QUICK_TEST else [11],
    "primary_loss_evaluation": "QLIKE",
    "secondary_metrics": ["RMSE", "MAE"],
    "dm_test": "HAC-adjusted",
    "mcs_alpha": MCS_ALPHA,
    "quick_test": QUICK_TEST,
    "tensorflow_version": tf.__version__,
    "python_version": platform.python_version(),
    "modules": {
        "capacity_match": RUN_CAPACITY_MATCH,
        "horizon_robustness": RUN_HORIZON_ROBUSTNESS,
        "regime_robustness": RUN_REGIME_ROBUSTNESS,
        "signed_return_ablation": RUN_SIGNED_RETURN_ABLATION,
        "equal_weight_SI": RUN_EQUAL_WEIGHT_SI,
        "parkinson_proxy_SI": RUN_PARKINSON_PROXY_SI,
        "local_global_SI": RUN_LOCAL_GLOBAL_SI,
        "learning_curves_SI": RUN_LEARNING_CURVES_SI,
        "thin_trading_SI": RUN_THIN_TRADING_SI,
        "extreme_return_SI": RUN_EXTREME_RETURN_SI,
        "expanding_window_SI": RUN_EXPANDING_WINDOW_SI,
        "lookback_ablation_SI": RUN_LOOKBACK_ABLATION_SI,
        "stock_level_SI": RUN_STOCK_LEVEL_SI
    }
}

with open(OUTPUT_ROOT / "00_manifest" / "run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

# ----------------------------------------------------------------------
# 31. FINAL OUTPUT INVENTORY
# ----------------------------------------------------------------------
inventory = []
for f in sorted(OUTPUT_ROOT.rglob("*")):
    if f.is_file():
        inventory.append({
            "relative_path": str(f.relative_to(OUTPUT_ROOT)),
            "size_kb": round(f.stat().st_size / 1024, 2)
        })
pd.DataFrame(inventory).to_csv(
    OUTPUT_ROOT / "00_manifest" / "output_inventory.csv", index=False
)

# A short run note
with open(OUTPUT_ROOT / "00_manifest" / "README_OUTPUTS.txt", "w") as f:
    f.write(
"""MAIN MANUSCRIPT OUTPUTS
-----------------------
02_main_results/main_accuracy.csv
02_main_results/DM_pairwise_QLIKE.csv
02_main_results/MCS_summary.csv
02_main_results/model_sector_interaction.txt
02_main_results/best_model_by_sector.csv
02_main_results/relative_loss_by_sector.csv
03_robustness/capacity_matched_accuracy.csv
03_robustness/horizon_robustness_all_horizons.csv
03_robustness/volatility_regime_accuracy.csv
04_ablations/signed_return_ablation.csv
06_figures/*

SUPPLEMENTARY INFORMATION OUTPUTS
---------------------------------
05_SI/random_seed_stability.csv
05_SI/tuning_results.csv
05_SI/chosen_hyperparameters.json
05_SI/equal_weighted_sector_accuracy.csv
05_SI/value_vs_equal_rankings.csv
05_SI/parkinson_proxy_accuracy.csv
05_SI/thin_trading_filtered_accuracy.csv
05_SI/extreme_return_winsorized_accuracy.csv
05_SI/learning_curves.csv
05_SI/local_vs_global_accuracy.csv
plus optional heavy-SI modules if enabled.

PRIMARY DATA RULE
-----------------
Main analysis uses lagged market-capitalization-weighted sector returns.
Weights at date t use market capitalization from t-1.
Equal-weighted sector results are a robustness check in SI.

FORECAST TARGET
---------------
Historical sector returns are inputs.
Future realized variance is the volatility target.
No return forecasting is performed.

CRASH-SAFE EXECUTION
--------------------
Outputs and checkpoints are stored persistently in Google Drive when run in Colab.
The script checkpoints tuning and every completed seed forecast. Re-running the
script resumes completed jobs instead of starting from the beginning.

HORIZON RULE
------------
Primary horizon h=5 is estimated once. Horizon robustness estimates only h=1
and h=22, then reuses saved h=5 primary seed forecasts to report h=1,5,22.
"""
    )

# ----------------------------------------------------------------------
# 32. ZIP EVERYTHING INTO ONE FILE AND DOWNLOAD
# ----------------------------------------------------------------------
# OUTPUT_ROOT is already persistent (Google Drive in Colab). The final ZIP is
# also created in persistent ZIP_ROOT, then downloaded to the browser.
ZIP_BASE = ZIP_ROOT / f"DL_Volatility_Forecasting_All_Outputs_{RUN_MODE}"
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", root_dir=OUTPUT_ROOT)

print("\n" + "="*72)
print("ANALYSIS COMPLETE")
print("Persistent output folder:", OUTPUT_ROOT)
print("Persistent checkpoint folder:", CHECKPOINT_ROOT)
print("Final ZIP:", zip_path)
print("="*72)

if IN_COLAB and AUTO_DOWNLOAD_ZIP:
    files.download(zip_path)


Mounting Google Drive for crash-safe persistent storage...
Mounted at /content/drive
Persistent project root: /content/drive/MyDrive/DL_Volatility_Forecasting_Project
Current run root      : /content/drive/MyDrive/DL_Volatility_Forecasting_Project/runs/v2_crashsafe_20260911_full_run
Resume enabled        : True
Additional horizons   : [1, 22] (h=5 will be reused from primary)
Using persistent data copy: /content/drive/MyDrive/DL_Volatility_Forecasting_Project/data/Final Master File_All variables 01082016-31072026.csv
Using data: /content/drive/MyDrive/DL_Volatility_Forecasting_Project/data/Final Master File_All variables 01082016-31072026.csv
Primary sectors (>= 3 stocks): ['Commercial and services', 'Banking', 'Manufacturing and Allied', 'Insurance', 'Agriculture', 'Energy and Petroleum', 'Investment']

PRIMARY | Commercial and services | h=5
   LSTM
    [resume] tuning/config loaded
    [resume] seed 11
    [resume] seed 29
    [resume] seed 47
    [resume] seed 71
    [resume] seed 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# ======================================================================
# LIGHTWEIGHT POST-PROCESSING FOR DL VOLATILITY PAPER — CORRECTED V2
# No TensorFlow training. Uses saved outputs only.
#
# Produces:
# 1) Holm-adjusted pairwise DM p-values within each sector
# 2) Corrected moving-block residual bootstrap test of Model x Sector interaction
# 3) Moving-block bootstrap confidence intervals for relative QLIKE losses
# 4) One ZIP containing all post-processing outputs
# ======================================================================

import os, sys, json, shutil, zipfile, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------
# 0. Install/import lightweight packages
# ----------------------------------------------------------------------
def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install(["numpy", "pandas", "scipy", "statsmodels"])

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ----------------------------------------------------------------------
# 1. Settings
# ----------------------------------------------------------------------
N_BOOT = 5000
BLOCK_LENGTH = 10      # block length for relative-QLIKE confidence intervals
INTERACTION_BLOCK_LENGTHS = [5, 10, 22]  # sensitivity for joint Model x Sector test
ALPHA = 0.05
SEED = 20260916

WORK = Path("/content/postprocessing_corrected" if IN_COLAB else "./postprocessing_corrected")
EXTRACT = WORK / "extracted"
OUT = WORK / "results"

# Start from a clean post-processing output directory. This prevents an older,
# flawed interaction-bootstrap table from being mixed into a corrected run.
if OUT.exists():
    shutil.rmtree(OUT)
EXTRACT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)

# ----------------------------------------------------------------------
# 2. Locate/upload the completed full-run ZIP
# ----------------------------------------------------------------------
def find_or_upload_zip():
    candidates = list(Path("/content").glob("DL_Volatility_Forecasting_All_Outputs*.zip")) if IN_COLAB else list(Path(".").glob("*.zip"))
    if candidates:
        return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]

    if IN_COLAB:
        print("Upload: DL_Volatility_Forecasting_All_Outputs_full_run.zip")
        uploaded = files.upload()
        zips = [Path("/content") / name for name in uploaded if name.lower().endswith(".zip")]
        if not zips:
            raise FileNotFoundError("No ZIP file uploaded.")
        return zips[0]

    raise FileNotFoundError("Place the completed output ZIP in the working directory.")

ZIP_PATH = find_or_upload_zip()
print("Using ZIP:", ZIP_PATH)

# Extract cleanly
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT)

# ----------------------------------------------------------------------
# 3. Find required saved-output files recursively
# ----------------------------------------------------------------------
def find_one(filename):
    hits = list(EXTRACT.rglob(filename))
    if not hits:
        raise FileNotFoundError(f"Could not find {filename} in ZIP.")
    if len(hits) > 1:
        print(f"WARNING: Multiple {filename} files found; using {hits[0]}")
    return hits[0]

PRED_FILE = find_one("main_test_predictions.csv")
DM_FILE = find_one("DM_pairwise_QLIKE.csv")

pred = pd.read_csv(PRED_FILE)
dm = pd.read_csv(DM_FILE)

pred["Date"] = pd.to_datetime(pred["Date"])

required_pred = {"Date", "sector", "model", "qlike_loss"}
missing = required_pred - set(pred.columns)
if missing:
    raise ValueError(f"Prediction file missing columns: {missing}")

print(f"Predictions: {len(pred):,} rows")
print(f"Sectors: {pred['sector'].nunique()}")
print(f"Models: {pred['model'].nunique()}")
print(f"DM rows: {len(dm)}")

# ----------------------------------------------------------------------
# 4. HOLM-ADJUSTED DM TESTS WITHIN EACH SECTOR
# ----------------------------------------------------------------------
dm_out = dm.copy()

# Flexible p-value column detection
p_candidates = [c for c in dm_out.columns if c.lower() in {"p_value", "pvalue", "p", "dm_p_value"}]
if not p_candidates:
    raise ValueError("Could not identify DM p-value column.")
pcol = p_candidates[0]

dm_out["holm_p_value"] = np.nan
dm_out["holm_reject_5pct"] = False

for sector, idx in dm_out.groupby("sector").groups.items():
    idx = list(idx)
    pvals = dm_out.loc[idx, pcol].astype(float).to_numpy()
    reject, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method="holm")
    dm_out.loc[idx, "holm_p_value"] = p_adj
    dm_out.loc[idx, "holm_reject_5pct"] = reject

dm_out["raw_reject_5pct"] = dm_out[pcol] < ALPHA

dm_out.to_csv(OUT / "Table_Holm_Adjusted_DM.csv", index=False)

holm_summary = (
    dm_out.groupby("sector")
          .agg(
              n_pairwise_tests=(pcol, "size"),
              n_raw_significant=("raw_reject_5pct", "sum"),
              n_holm_significant=("holm_reject_5pct", "sum")
          )
          .reset_index()
)
holm_summary.to_csv(OUT / "Table_Holm_DM_Summary_by_Sector.csv", index=False)

# ----------------------------------------------------------------------
# 5. PREPARE BALANCED DATE x (SECTOR,MODEL) LOSS PANEL
# ----------------------------------------------------------------------
# Same-date resampling preserves contemporaneous cross-sector/model dependence.
wide = pred.pivot_table(
    index="Date",
    columns=["sector", "model"],
    values="qlike_loss",
    aggfunc="mean"
).sort_index()

# For the joint bootstrap, use dates with complete losses across all cells.
wide_complete = wide.dropna(axis=0, how="any").copy()

if len(wide_complete) < 50:
    raise ValueError(
        f"Only {len(wide_complete)} complete dates available for joint bootstrap."
    )

print(f"Complete dates for joint block bootstrap: {len(wide_complete)}")

# Long-form complete panel
joint_long = (
    wide_complete.stack(["sector", "model"], future_stack=True)
                 .rename("qlike_loss")
                 .reset_index()
)

# ----------------------------------------------------------------------
# 6. OBSERVED MODEL x SECTOR INTERACTION EFFECT
# ----------------------------------------------------------------------
# Reduced model = additive model only
#                qlike_loss ~ model + sector
# Full model    = additive model + model x sector interaction
#                qlike_loss ~ model * sector
#
# The null hypothesis is that there is NO Model x Sector interaction.
# The bootstrap must therefore generate samples under that restricted null.

reduced_formula = "qlike_loss ~ C(model) + C(sector)"
full_formula = "qlike_loss ~ C(model) * C(sector)"

reduced_fit = smf.ols(reduced_formula, data=joint_long).fit()
full_fit = smf.ols(full_formula, data=joint_long).fit()

rss_reduced = float(np.sum(reduced_fit.resid ** 2))
rss_full = float(np.sum(full_fit.resid ** 2))
delta_rss_obs = rss_reduced - rss_full

# Conventional nested-model F statistic is retained only as a reference.
f_stat_obs, f_p_classical, df_diff = full_fit.compare_f_test(reduced_fit)

# ----------------------------------------------------------------------
# 7. CORRECTED MOVING-BLOCK RESIDUAL BOOTSTRAP UNDER THE NULL
# ----------------------------------------------------------------------
# IMPORTANT CORRECTION
# --------------------
# A residual bootstrap under the additive/null model must REMOVE any
# sector x model cell-specific mean remaining in the restricted residuals.
#
# If those cell-specific residual means are resampled without centering,
# the bootstrap pseudo-samples retain part of the observed interaction
# signal and therefore do NOT correctly impose H0.
#
# We therefore:
#   1) fit the additive restricted model,
#   2) arrange its fitted values/residuals as Date x (Sector,Model),
#   3) center the restricted residuals WITHIN every Sector x Model cell,
#   4) jointly resample whole residual date-vectors in moving blocks,
#   5) add the resampled centered residuals to restricted fitted values,
#   6) recompute the additive-vs-interaction RSS improvement.
#
# Joint date-vector resampling preserves contemporaneous dependence across
# sectors/models; moving blocks preserve serial dependence.

dates = wide_complete.index.to_numpy()
cells = list(wide_complete.columns)
T = len(dates)

tmp = joint_long.copy()
tmp["fitted_null"] = reduced_fit.fittedvalues
tmp["restricted_resid"] = reduced_fit.resid

fit_wide = (
    tmp.pivot(index="Date", columns=["sector", "model"], values="fitted_null")
       .loc[wide_complete.index, cells]
)
resid_wide = (
    tmp.pivot(index="Date", columns=["sector", "model"], values="restricted_resid")
       .loc[wide_complete.index, cells]
)

fit_mat = fit_wide.to_numpy(dtype=float)
resid_mat = resid_wide.to_numpy(dtype=float)

# KEY FIX: impose the no-interaction null by removing each cell's residual mean.
resid_null = resid_mat - np.mean(resid_mat, axis=0, keepdims=True)

# Numerical check: each cell's null residual mean should be effectively zero.
max_abs_cell_mean = float(np.max(np.abs(np.mean(resid_null, axis=0))))
if max_abs_cell_mean > 1e-10:
    raise RuntimeError(
        f"Null residual centering failed: max cell mean = {max_abs_cell_mean}"
    )

def moving_block_indices(T, block_len, rng):
    out = []
    while len(out) < T:
        start = int(rng.integers(0, T))
        block = [(start + j) % T for j in range(block_len)]
        out.extend(block)
    return np.array(out[:T], dtype=int)

interaction_rows = []
interaction_distributions = {}

for block_len in INTERACTION_BLOCK_LENGTHS:
    brng = np.random.default_rng(SEED)
    boot_delta = np.empty(N_BOOT)

    for b in range(N_BOOT):
        idx = moving_block_indices(T, block_len, brng)

        # Resample whole date residual-vectors so dependence across all
        # sector/model cells on a given date is retained.
        y_star_mat = fit_mat + resid_null[idx, :]

        y_star_wide = pd.DataFrame(
            y_star_mat,
            index=wide_complete.index,
            columns=wide_complete.columns
        )

        y_star = (
            y_star_wide.stack(["sector", "model"], future_stack=True)
                       .rename("qlike_loss")
                       .reset_index()
        )

        rfit = smf.ols(reduced_formula, data=y_star).fit()
        ffit = smf.ols(full_formula, data=y_star).fit()

        rss_r = float(np.sum(rfit.resid ** 2))
        rss_f = float(np.sum(ffit.resid ** 2))
        boot_delta[b] = rss_r - rss_f

    bootstrap_p = (
        1 + np.sum(boot_delta >= delta_rss_obs)
    ) / (N_BOOT + 1)

    interaction_rows.append({
        "n_complete_dates": T,
        "n_sectors": pred["sector"].nunique(),
        "n_models": pred["model"].nunique(),
        "block_length": block_len,
        "bootstrap_replications": N_BOOT,
        "observed_delta_RSS_reduced_minus_full": delta_rss_obs,
        "classical_nested_F_reference_only": float(f_stat_obs),
        "classical_F_p_value_reference_only": float(f_p_classical),
        "df_difference": float(df_diff),
        "max_abs_centered_cell_residual_mean": max_abs_cell_mean,
        "moving_block_bootstrap_p_value": float(bootstrap_p),
        "reject_no_model_x_sector_interaction_5pct": bool(bootstrap_p < ALPHA),
        "null_imposition":
            "restricted residuals centered within each sector-model cell"
    })

    interaction_distributions[f"block_{block_len}"] = boot_delta

interaction_summary = pd.DataFrame(interaction_rows)

interaction_summary.to_csv(
    OUT / "Table_CORRECTED_BlockBootstrap_Model_x_Sector_Interaction.csv",
    index=False
)

pd.DataFrame(interaction_distributions).to_csv(
    OUT / "SI_CORRECTED_BlockBootstrap_Interaction_Distributions.csv",
    index=False
)

# ----------------------------------------------------------------------
# 8. BLOCK-BOOTSTRAP CIs FOR RELATIVE QLIKE DIFFERENCES
# ----------------------------------------------------------------------
# For each sector:
#  - identify observed best model using mean QLIKE
#  - compare every model with that benchmark
#  - resample common dates in blocks
#  - CI for mean(model loss - benchmark loss)
#
# Positive difference = model has higher (worse) QLIKE than benchmark.

ci_rows = []

for sector, g in pred.groupby("sector"):
    sw = g.pivot_table(
        index="Date", columns="model", values="qlike_loss", aggfunc="mean"
    ).sort_index().dropna(axis=0, how="any")

    if len(sw) < 30:
        continue

    mean_losses = sw.mean()
    best_model = mean_losses.idxmin()
    best_loss = float(mean_losses.min())

    arr = sw.to_numpy(dtype=float)
    models = list(sw.columns)
    best_idx = models.index(best_model)
    Ts = len(sw)

    boot_means = {m: np.empty(N_BOOT) for m in models}

    # Use a separate reproducible sector generator
    sector_seed = SEED + sum(ord(ch) for ch in str(sector))
    srng = np.random.default_rng(sector_seed)

    for b in range(N_BOOT):
        idx = moving_block_indices(Ts, BLOCK_LENGTH, srng)
        xb = arr[idx, :]
        base = xb[:, best_idx]

        for j, model in enumerate(models):
            boot_means[model][b] = np.mean(xb[:, j] - base)

    for j, model in enumerate(models):
        diff = float(mean_losses[model] - best_loss)
        ci_low, ci_high = np.quantile(
            boot_means[model], [ALPHA/2, 1-ALPHA/2]
        )

        relative_ratio = float(mean_losses[model] / best_loss) if best_loss > 0 else np.nan
        excess_pct = 100.0 * (relative_ratio - 1.0) if np.isfinite(relative_ratio) else np.nan

        ci_rows.append({
            "sector": sector,
            "benchmark_observed_best_model": best_model,
            "model": model,
            "mean_QLIKE_model": float(mean_losses[model]),
            "mean_QLIKE_best": best_loss,
            "mean_QLIKE_difference_model_minus_best": diff,
            "bootstrap_95CI_low_difference": float(ci_low),
            "bootstrap_95CI_high_difference": float(ci_high),
            "CI_excludes_zero": bool((ci_low > 0) or (ci_high < 0)),
            "relative_QLIKE_ratio": relative_ratio,
            "percent_excess_QLIKE_vs_best": excess_pct,
            "n_dates": Ts
        })

ci_table = pd.DataFrame(ci_rows)
ci_table.to_csv(
    OUT / "Table_BlockBootstrap_Relative_QLIKE_CIs.csv",
    index=False
)

# Compact manuscript-oriented version
manuscript_ci = ci_table[
    [
        "sector",
        "benchmark_observed_best_model",
        "model",
        "mean_QLIKE_model",
        "mean_QLIKE_difference_model_minus_best",
        "bootstrap_95CI_low_difference",
        "bootstrap_95CI_high_difference",
        "percent_excess_QLIKE_vs_best",
        "CI_excludes_zero"
    ]
].copy()

manuscript_ci.to_csv(
    OUT / "Table_Manuscript_Relative_QLIKE_CIs.csv",
    index=False
)

# ----------------------------------------------------------------------
# 9. HUMAN-READABLE SUMMARY
# ----------------------------------------------------------------------
lines = []
lines.append("LIGHTWEIGHT POST-PROCESSING SUMMARY")
lines.append("=" * 70)
lines.append("")
lines.append("HOLM-ADJUSTED DM TESTS")
lines.append(holm_summary.to_string(index=False))
lines.append("")
lines.append("CORRECTED MODEL x SECTOR MOVING-BLOCK BOOTSTRAP")
lines.append(interaction_summary.to_string(index=False))
lines.append("")
lines.append(
    "IMPORTANT: restricted-model residuals were centered within each "
    "Sector x Model cell before resampling so the bootstrap correctly "
    "imposes the no-interaction null."
)
lines.append("")
lines.append("RELATIVE QLIKE BOOTSTRAP")
lines.append(
    "Positive model-minus-best difference means worse QLIKE than the observed sector-best model."
)
lines.append(
    "Do not interpret the observed best model as universally dominant when the CI includes zero."
)
lines.append("")
lines.append("METHOD NOTES")
lines.append(f"- Bootstrap replications: {N_BOOT}")
lines.append(f"- Relative-QLIKE CI block length: {BLOCK_LENGTH}")
lines.append(f"- Interaction-test block lengths: {INTERACTION_BLOCK_LENGTHS}")
lines.append("- Entire date vectors are resampled together to preserve contemporaneous dependence.")
lines.append("- Joint interaction bootstrap is generated under the additive no-interaction null.")
lines.append("- Restricted residuals are centered within each Sector x Model cell before block resampling.")
lines.append("- No neural-network model is re-estimated in this script.")

summary_path = OUT / "CORRECTED_Postprocessing_Summary.txt"
summary_path.write_text("\n".join(lines), encoding="utf-8")

print("\n" + "\n".join(lines))

# ----------------------------------------------------------------------
# 10. ZIP RESULTS AND DOWNLOAD
# ----------------------------------------------------------------------
zip_base = WORK / "DL_Volatility_Postprocessing_Results_CORRECTED"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUT)

print("\nCreated:", zip_path)

if IN_COLAB:
    files.download(zip_path)


Using ZIP: /content/DL_Volatility_Forecasting_All_Outputs_full_run.zip
Predictions: 13,720 rows
Sectors: 7
Models: 4
DM rows: 42
Complete dates for joint block bootstrap: 490

LIGHTWEIGHT POST-PROCESSING SUMMARY

HOLM-ADJUSTED DM TESTS
                  sector  n_pairwise_tests  n_raw_significant  n_holm_significant
             Agriculture                 6                  1                   1
                 Banking                 6                  0                   0
 Commercial and services                 6                  5                   5
    Energy and Petroleum                 6                  2                   0
               Insurance                 6                  1                   0
              Investment                 6                  4                   3
Manufacturing and Allied                 6                  2                   0

CORRECTED MODEL x SECTOR MOVING-BLOCK BOOTSTRAP
 n_complete_dates  n_sectors  n_models  block_length  boots

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>